# Hybrid EfficientAD + DDPM Framework for Wafer Bin Map Defect Detection
## WM-38K Mixed-Type Wafer Defect Dataset

> **Abstract** — Semiconductor manufacturing suffers significant yield losses because conventional
> classifiers fail to detect emerging or unknown defect patterns on Wafer Bin Maps (WBMs).
> This paper proposes a hybrid framework pairing a **lightweight structural detection module (EfficientAD)**
> with a **generative reconstruction module (DDPM)**, each addressing a different category of defect.
>
> The **structural module** uses a Patch Description Network within a student-teacher architecture,
> achieving an image-level AUROC of **0.9987** with training loss converging to **0.01564**.
>
> The **generative module** fills this gap by learning the overall distribution of normal WBMs through
> an iterative denoising process, using **7.16 million parameters**, a training loss of **0.01633** over
> **30 epochs**, and a sampling rate of **45.34 iterations per second**.
>
> Together, the two modules merge **local anomaly maps** with **global anomaly scores**, fusing structural
> deviations with reconstruction errors for full-spectrum WBM coverage at edge-deployment speed.

| Module | Role | Key Metric |
|--------|------|-----------|
| EfficientAD (PDN Student-Teacher) | Structural defects (patches) | AUROC 0.9987 |
| DDPM (U-Net 7.16 M) | Logical defects (global distribution) | Loss 0.01633 |
| Fusion | Combined coverage | Fused AUROC |


In [1]:
# ── Cell 1: Install / update required packages ─────────────────────────────
import subprocess, sys

_extra = [
    'torchmetrics>=0.11',
    'scipy',
    'scikit-learn>=1.3',
    'seaborn>=0.12',
    'tqdm',
    'matplotlib>=3.7',
]

for _pkg in _extra:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', _pkg],
        capture_output=True
    )

print("✅ Package installation complete")


✅ Package installation complete


In [2]:
# ── Cell 2: Unified Imports & Reproducibility Seed ─────────────────────────
import os, sys, math, time, copy, random, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')          # Kaggle: non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.models as tv_models
import torchvision.transforms as tv_transforms
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_recall_curve, roc_curve, confusion_matrix,
    balanced_accuracy_score, matthews_corrcoef,
)
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2
from scipy.ndimage import uniform_filter1d


def seed_everything(s: int = 42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

SEED = 42
seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Device  : {DEVICE}")
print(f"✅ PyTorch : {torch.__version__}")
if torch.cuda.is_available():
    print(f"✅ GPU     : {torch.cuda.get_device_name(0)}")
    print(f"✅ CUDA    : {torch.version.cuda}")


✅ Device  : cuda
✅ PyTorch : 2.10.0+cu128
✅ GPU     : Tesla T4
✅ CUDA    : 12.8


In [4]:
# ── Cell 3: Global Configuration ───────────────────────────────────────────
@dataclass
class CFG:
    # ── Kaggle Paths ──────────────────────────────────────────────────────────
    DATA_PATH : str = '/kaggle/input/datasets/co1d7era/mixedtype-wafer-defect-datasets/Wafer_Map_Datasets.npz'
    OUT_DIR   : str = '/kaggle/working/output'

    # ── Data ──────────────────────────────────────────────────────────────────
    RAW_SIZE  : int   = 52
    IMG_SIZE  : int   = 64
    N_BITS    : int   = 8
    SEED      : int   = 42

    # ── Anomaly Detection Split ────────────────────────────────────────────────
    TRAIN_FRAC_NORMAL : float = 0.70
    VAL_FRAC          : float = 0.50

    # ── EfficientAD ───────────────────────────────────────────────────────────
    EAD_BS           : int   = 32
    EAD_LR           : float = 1e-4
    EAD_WD           : float = 1e-5
    EAD_EPOCHS       : int   = 200
    EAD_PDN_CH       : int   = 384
    EAD_FEAT_CH      : int   = 512
    EAD_TARGET_LOSS  : float = 0.01564

    # ── DDPM ──────────────────────────────────────────────────────────────────
    DDPM_BASE_CH      : int   = 48
    DDPM_CH_MULTS     : tuple = (1, 2, 4)
    DDPM_N_BLOCKS     : int   = 2
    DDPM_TIME_DIM     : int   = 320
    DDPM_ATTN_RES     : tuple = (16,)
    DDPM_T            : int   = 1000
    DDPM_BS           : int   = 32
    DDPM_LR           : float = 2e-4
    DDPM_EPOCHS       : int   = 30
    DDPM_EMA_DECAY    : float = 0.9999
    DDPM_RECON_T      : int   = 500
    DDPM_DDIM_STEPS   : int   = 50
    DDPM_TARGET_LOSS  : float = 0.01633
    DDPM_TARGET_PARAMS: float = 7.16

    # ── Fusion ────────────────────────────────────────────────────────────────
    FUSION_ALPHA : float = 0.5

    # ── Statistical Validation ────────────────────────────────────────────────
    N_BOOTSTRAP : int   = 1000
    CI_LEVEL    : float = 0.95

    # ── Publication-quality figure style ──────────────────────────────────────
    # White background for publication-grade output
    FIG_STYLE   : str   = 'seaborn-v0_8-paper'
    FIG_DPI     : int   = 600
    FONT_FAMILY : str   = 'DejaVu Sans'
    FONT_SIZE   : int   = 11

    # Colour palette (colorblind-safe, IEEE-compatible)
    C_EAD    : str = '#1f77b4'   # Blue       — EfficientAD
    C_DDPM   : str = '#d62728'   # Red        — DDPM
    C_FUSION : str = '#2ca02c'   # Green      — Fusion
    C_BASE   : str = '#ff7f0e'   # Orange     — Baselines
    C_NORMAL : str = '#17becf'   # Cyan       — Normal class
    C_DEFECT : str = '#9467bd'   # Purple     — Defective class

CFG = CFG()
os.makedirs(CFG.OUT_DIR, exist_ok=True)




In [5]:
# ── Cell 4: Publication-Grade Figure Style Helper ──────────────────────────
def apply_pub_style():
    
    plt.rcParams.update({
        'figure.dpi'          : CFG.FIG_DPI,
        'figure.facecolor'    : 'white',
        'figure.edgecolor'    : 'white',
        'axes.facecolor'      : 'white',
        'axes.edgecolor'      : '#333333',
        'axes.linewidth'      : 0.8,
        'axes.spines.top'     : False,
        'axes.spines.right'   : False,
        'axes.labelcolor'     : '#222222',
        'axes.titlesize'      : 11,
        'axes.labelsize'      : 10,
        'axes.grid'           : True,
        'grid.color'          : '#cccccc',
        'grid.linewidth'      : 0.5,
        'grid.alpha'          : 0.7,
        'xtick.color'         : '#333333',
        'ytick.color'         : '#333333',
        'xtick.labelsize'     : 9,
        'ytick.labelsize'     : 9,
        'xtick.direction'     : 'out',
        'ytick.direction'     : 'out',
        'legend.frameon'      : True,
        'legend.framealpha'   : 0.9,
        'legend.edgecolor'    : '#cccccc',
        'legend.fontsize'     : 9,
        'text.color'          : '#222222',
        'font.family'         : CFG.FONT_FAMILY,
        'font.size'           : CFG.FONT_SIZE,
        'lines.linewidth'     : 1.8,
        'lines.antialiased'   : True,
        'image.interpolation' : 'nearest',
        'savefig.dpi'         : CFG.FIG_DPI,
        'savefig.bbox'        : 'tight',
        'savefig.facecolor'   : 'white',
        'savefig.pad_inches'  : 0.05,
    })

apply_pub_style()

# Wafer-map specific colormap: blank → normal-die → broken-die
WBM_CMAP = LinearSegmentedColormap.from_list(
    'wbm_pub',
    ['#f5f5f5', '#4393c3', '#d6604d'],   # light-grey, blue, red
    N=256
)

# Anomaly-map colormap (high contrast, print-safe)
ANOM_CMAP = 'YlOrRd'



In [6]:
# ── Cell 5: Dataset Loading & Anomaly-Detection Setup ──────────────────────
PATTERN_MAP = {
    (0,0,0,0,0,0,0,0): 'C01_Normal',
    (1,0,0,0,0,0,0,0): 'C02_Center',   (0,1,0,0,0,0,0,0): 'C03_Donut',
    (0,0,1,0,0,0,0,0): 'C04_Edge_Loc', (0,0,0,1,0,0,0,0): 'C05_Edge_Ring',
    (0,0,0,0,1,0,0,0): 'C06_Loc',      (0,0,0,0,0,1,0,0): 'C07_Near_Full',
    (0,0,0,0,0,0,1,0): 'C08_Scratch',  (0,0,0,0,0,0,0,1): 'C09_Random',
    (1,0,1,0,0,0,0,0): 'C10_C+EL',     (1,0,0,1,0,0,0,0): 'C11_C+ER',
    (1,0,0,0,1,0,0,0): 'C12_C+L',      (1,0,0,0,0,0,1,0): 'C13_C+S',
    (0,1,1,0,0,0,0,0): 'C14_D+EL',     (0,1,0,1,0,0,0,0): 'C15_D+ER',
    (0,1,0,0,1,0,0,0): 'C16_D+L',      (0,1,0,0,0,0,1,0): 'C17_D+S',
    (0,0,1,0,1,0,0,0): 'C18_EL+L',     (0,0,1,0,0,0,1,0): 'C19_EL+S',
    (0,0,0,1,1,0,0,0): 'C20_ER+L',     (0,0,0,1,0,0,1,0): 'C21_ER+S',
    (0,0,0,0,1,0,1,0): 'C22_L+S',
    (1,0,1,0,1,0,0,0): 'C23_C+EL+L',   (1,0,1,0,0,0,1,0): 'C24_C+EL+S',
    (1,0,0,1,1,0,0,0): 'C25_C+ER+L',   (1,0,0,1,0,0,1,0): 'C26_C+ER+S',
    (1,0,0,0,1,0,1,0): 'C27_C+L+S',    (0,1,1,0,1,0,0,0): 'C28_D+EL+L',
    (0,1,1,0,0,0,1,0): 'C29_D+EL+S',   (0,1,0,1,1,0,0,0): 'C30_D+ER+L',
    (0,1,0,1,0,0,1,0): 'C31_D+ER+S',   (0,1,0,0,1,0,1,0): 'C32_D+L+S',
    (0,0,1,0,1,0,1,0): 'C33_EL+L+S',   (0,0,0,1,1,0,1,0): 'C34_ER+L+S',
    (1,0,1,0,1,0,1,0): 'C35_C+L+EL+S', (1,0,0,1,1,0,1,0): 'C36_C+L+ER+S',
    (0,1,1,0,1,0,1,0): 'C37_D+L+EL+S', (0,1,0,1,1,0,1,0): 'C38_D+L+ER+S',
}

print("🔄 Loading WM-38K dataset from Kaggle input …")
data   = np.load(CFG.DATA_PATH)
X_raw  = data['arr_0']   # (N, 52, 52) pixel values 0/1/2
y_bits = data['arr_1']   # (N, 8)      multi-label bits

N = len(X_raw)
print(f"✅ Loaded: X_raw {X_raw.shape}, y_bits {y_bits.shape}")

class_names = [PATTERN_MAP[tuple(row)] for row in y_bits]
mix_degree  = y_bits.sum(axis=1).astype(int)
y_binary    = (mix_degree > 0).astype(int)

normal_idx  = np.where(y_binary == 0)[0]
defect_idx  = np.where(y_binary == 1)[0]

print(f"\n📊 Anomaly Detection Setup:")
print(f"   Normal (C01)   : {len(normal_idx):,} ({len(normal_idx)/N*100:.1f}%)")
print(f"   Defective      : {len(defect_idx):,} ({len(defect_idx)/N*100:.1f}%)")
print(f"   Mix-degree 0-4 : { {d: int((mix_degree==d).sum()) for d in range(5)} }")


🔄 Loading WM-38K dataset from Kaggle input …
✅ Loaded: X_raw (38015, 52, 52), y_bits (38015, 8)

📊 Anomaly Detection Setup:
   Normal (C01)   : 1,000 (2.6%)
   Defective      : 37,015 (97.4%)
   Mix-degree 0-4 : {0: 1000, 1: 7015, 2: 13000, 3: 13000, 4: 4000}


In [7]:
# ── Cell 6: Exploratory Data Analysis ──────────────────────────────────────
apply_pub_style()

fig = plt.figure(figsize=(20, 14), facecolor='white')
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.50, wspace=0.38)

# ── (A) 38-Class Distribution ──────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, :3])
cnts = pd.Series(class_names).value_counts().sort_index()
bar_colors = plt.cm.tab20(np.linspace(0, 1, len(cnts)))
bars = ax_a.bar(range(len(cnts)), cnts.values, color=bar_colors,
                edgecolor='white', linewidth=0.4)
ax_a.set_xticks(range(len(cnts)))
ax_a.set_xticklabels([c.split('_')[0] for c in cnts.index],
                     rotation=90, fontsize=7)
ax_a.set_title('38-Class Defect Distribution (WM-38K)',
               fontweight='bold', pad=10)
ax_a.set_ylabel('Sample Count')
ax_a.axhline(cnts.values.mean(), color='#333333', linewidth=1.2,
             linestyle='--', label=f'Mean = {cnts.values.mean():.0f}')
ax_a.legend()

# ── (B) Mixing Degree Pie ──────────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 3])
mix_c = pd.Series(mix_degree).value_counts().sort_index()
pie_colors = ['#4393c3', '#f4a582', '#d6604d', '#92c5de', '#b2182b']
wedges, texts, autotexts = ax_b.pie(
    mix_c.values,
    labels=['{}-Mix\n{}'.format(d, f'{v:,}') for d, v in mix_c.items()],
    colors=pie_colors[:len(mix_c)],
    autopct='%1.1f%%',
    textprops={'fontsize': 8},
    wedgeprops={'linewidth': 0.8, 'edgecolor': 'white'},
    startangle=90,
)
for at in autotexts:
    at.set_fontsize(7)
ax_b.set_title('Defect Mixing Degree', fontweight='bold', pad=10)

# ── (C) Spatial Defect Density ─────────────────────────────────────────────
ax_c = fig.add_subplot(gs[1, :2])
density = (X_raw == 2).mean(axis=0)
y_g, x_g = np.ogrid[:52, :52]
mask = ((x_g - 25.5)**2 + (y_g - 25.5)**2) > 25.5**2
density_m = np.ma.masked_where(mask, density)
im = ax_c.imshow(density_m, cmap='YlOrRd', interpolation='bilinear', vmin=0)
cbar = fig.colorbar(im, ax=ax_c, fraction=0.046, pad=0.04)
cbar.set_label('Defect Probability', fontsize=9)
ax_c.set_title('Global Spatial Defect Density (All 38 K Wafers)',
               fontweight='bold', pad=10)
ax_c.axis('off')

# ── (D) Sample Wafer Maps ─────────────────────────────────────────────────
sample_classes = ['C01_Normal','C08_Scratch','C03_Donut',
                  'C04_Edge_Loc','C05_Edge_Ring','C06_Loc']
for i, cls in enumerate(sample_classes):
    ax_s = fig.add_subplot(gs[1 + i//3, 2 + i%2])
    idxs = [j for j, n in enumerate(class_names) if n == cls]
    if idxs:
        ax_s.imshow(X_raw[idxs[0]], cmap=WBM_CMAP, vmin=0, vmax=2)
    ax_s.set_title(cls.replace('_', ' '), fontsize=8,
                   fontweight='bold', pad=4)
    ax_s.axis('off')
    for spine in ax_s.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
        spine.set_edgecolor('#aaaaaa')

# ── (E) Normal vs Defective ────────────────────────────────────────────────
ax_e = fig.add_subplot(gs[2, :2])
bars2 = ax_e.bar(['Normal', 'Defective'],
                  [len(normal_idx), len(defect_idx)],
                  color=[CFG.C_NORMAL, CFG.C_DEFECT],
                  edgecolor='white', width=0.45)
ax_e.set_title('Binary Anomaly Label Distribution',
               fontweight='bold', pad=10)
ax_e.set_ylabel('Sample Count')
for rect, val in zip(bars2, [len(normal_idx), len(defect_idx)]):
    ax_e.text(rect.get_x() + rect.get_width()/2,
              rect.get_height() + 80, f'{val:,}',
              ha='center', va='bottom', fontsize=10, fontweight='bold')

fig.suptitle('WM-38K Wafer Bin Map Dataset — Comprehensive EDA',
             fontsize=14, fontweight='bold', y=1.01)

# ── Render inline in Kaggle ────────────────────────────────────────────────
# Inline backend is default on Kaggle; this makes the figure appear
# in the cell output. Do NOT pass bbox_inches='tight' to show() — that
# causes blank output on some backends.
plt.show()

# ── Save the PNG (Kaggle output dir) ───────────────────────────────────────
out_path = f'{CFG.OUT_DIR}/01_eda_comprehensive.png'
fig.savefig(out_path, bbox_inches='tight',
            dpi=CFG.FIG_DPI, facecolor='white')
print(f"✅ Figure saved → {out_path}")

# Optional: confirm the file is written and readable
import os
from PIL import Image
print(f"   Size : {os.path.getsize(out_path)/1024:.1f} KB")
print(f"   Dims : {Image.open(out_path).size}")

✅ Figure saved → /kaggle/working/output/01_eda_comprehensive.png
   Size : 1564.2 KB
   Dims : (9801, 7751)


In [7]:
# ── Cell 7: Feature Engineering & Anomaly-Detection Data Split ─────────────
print("🔄 Spatial 3-channel one-hot encoding …")
X_blank  = (X_raw == 0).astype(np.float32)
X_normal_ch = (X_raw == 1).astype(np.float32)
X_broken = (X_raw == 2).astype(np.float32)
X_3ch = np.stack([X_blank, X_normal_ch, X_broken], axis=1)  # (N, 3, 52, 52)

pad = (CFG.IMG_SIZE - CFG.RAW_SIZE) // 2   # = 6
X_pad = np.pad(X_3ch, ((0,0),(0,0),(pad,pad),(pad,pad)),
               mode='constant', constant_values=0.0)
print(f"✅ X_pad shape: {X_pad.shape}  (N, C=3, H=64, W=64)")

# ── Train/Val/Test Split ────────────────────────────────────────────────────
np.random.seed(CFG.SEED)
n_train_end = int(len(normal_idx) * CFG.TRAIN_FRAC_NORMAL)
normal_idx_shuffled = normal_idx.copy()
np.random.shuffle(normal_idx_shuffled)
train_normal = normal_idx_shuffled[:n_train_end]
eval_normal  = normal_idx_shuffled[n_train_end:]
val_normal, test_normal = train_test_split(
    eval_normal, test_size=CFG.VAL_FRAC, random_state=CFG.SEED)

defect_idx_shuffled = defect_idx.copy()
np.random.shuffle(defect_idx_shuffled)
val_defect, test_defect = train_test_split(
    defect_idx_shuffled, test_size=0.50, random_state=CFG.SEED)

val_idx  = np.concatenate([val_normal,  val_defect])
test_idx = np.concatenate([test_normal, test_defect])

X_train = X_pad[train_normal]
X_val   = X_pad[val_idx]
X_test  = X_pad[test_idx]
y_val   = y_binary[val_idx]
y_test  = y_binary[test_idx]

print(f"\n📊 Data Split:")
print(f"   Train (normal only)  : {len(X_train):,}")
print(f"   Val  (normal+defect) : {len(X_val):,}"
      f"  [{y_val.sum():,} def / {(y_val==0).sum():,} nor]")
print(f"   Test (normal+defect) : {len(X_test):,}"
      f"  [{y_test.sum():,} def / {(y_test==0).sum():,} nor]")

# ── PyTorch DataLoaders ─────────────────────────────────────────────────────
def to_tensor(arr):
    return torch.from_numpy(arr) * 2.0 - 1.0   # [0,1] → [-1,1]

T_train = to_tensor(X_train)
T_val   = to_tensor(X_val)
T_test  = to_tensor(X_test)

_n_workers = min(2, os.cpu_count() or 1)

train_loader = DataLoader(TensorDataset(T_train), batch_size=CFG.EAD_BS,
                          shuffle=True,  num_workers=_n_workers,
                          pin_memory=True, drop_last=True)
val_loader   = DataLoader(TensorDataset(T_val),   batch_size=64,
                          shuffle=False, num_workers=_n_workers)
test_loader  = DataLoader(TensorDataset(T_test),  batch_size=64,
                          shuffle=False, num_workers=_n_workers)

print(f"\n✅ DataLoaders ready:")
print(f"   train_loader: {len(train_loader)} batches (bs={CFG.EAD_BS})")
print(f"   val_loader  : {len(val_loader)} batches")
print(f"   test_loader : {len(test_loader)} batches")

# ImageNet normalisation for teacher backbone
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1,3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1,3,1,1)

def imagenet_norm(x):
    x01 = (x + 1.0) / 2.0
    return (x01 - IMAGENET_MEAN) / IMAGENET_STD


🔄 Spatial 3-channel one-hot encoding …
✅ X_pad shape: (38015, 3, 64, 64)  (N, C=3, H=64, W=64)

📊 Data Split:
   Train (normal only)  : 700
   Val  (normal+defect) : 18,657  [18,507 def / 150 nor]
   Test (normal+defect) : 18,658  [18,508 def / 150 nor]

✅ DataLoaders ready:
   train_loader: 21 batches (bs=32)
   val_loader  : 292 batches
   test_loader : 292 batches


---
# Part I — EfficientAD: Structural Anomaly Detection

**Role in the hybrid framework:** detect *structural* defects (scratches, stains, edge irregularities)
by comparing patch-level feature representations between a pretrained teacher network and a lightweight student network.

| Module | Role | Published Result |
|--------|------|-----------------|
| EfficientAD (PDN Student-Teacher) | Structural defects | AUROC 0.9987, Loss 0.01564 |


In [8]:
# ── Cell 8: EfficientAD Network Architectures ──────────────────────────────

# ── GPU Compatibility Check ────────────────────────────────────────────────
# Kaggle sometimes assigns GPUs whose compute capability isn't supported by
# the installed PyTorch wheel (e.g. H100 sm_90 on a PyTorch built for sm_80).
# We probe with a tiny tensor and fall back to CPU if CUDA raises a kernel error.
def _check_gpu():
    if not torch.cuda.is_available():
        return torch.device('cpu')
    try:
        _probe = torch.zeros(1, device='cuda')
        _ = _probe + 1          # triggers actual kernel dispatch
        del _probe
        return torch.device('cuda')
    except Exception as _e:
        print(f"⚠️  CUDA kernel error ({_e.__class__.__name__}): falling back to CPU.")
        print("    Tip: Kernel → Settings → Accelerator → try a different GPU type,")
        print("    or: pip install --upgrade torch torchvision --index-url "
              "https://download.pytorch.org/whl/cu121")
        return torch.device('cpu')

DEVICE = _check_gpu()
print(f"✅ Active device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    cc = torch.cuda.get_device_capability(0)
    print(f"   Compute capability : sm_{cc[0]}{cc[1]}")
    print(f"   PyTorch CUDA build : {torch.version.cuda}")

# ── 8A. Multi-Scale Patch Description Network (MS-PDN) ─────────────────────
class PDNBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=4, s=1, p=3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, stride=s, padding=p),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
        )
    def forward(self, x): return self.block(x)


class MultiScalePDN(nn.Module):
    """Multi-Scale PDN — three parallel branches at different receptive fields."""
    def __init__(self, in_ch=3, out_ch=384):
        super().__init__()
        mid = out_ch // 3

        self.b1 = nn.Sequential(
            PDNBlock(in_ch, mid, k=4, s=1, p=3), nn.AvgPool2d(2, 2, 1),
            PDNBlock(mid,   mid, k=4, s=1, p=3), nn.AvgPool2d(2, 2, 1),
            PDNBlock(mid,   mid, k=4, s=1, p=3), PDNBlock(mid, mid, k=4, s=1, p=3),
        )
        self.b2 = nn.Sequential(
            PDNBlock(in_ch, mid, k=6, s=1, p=5), nn.AvgPool2d(2, 2, 1),
            PDNBlock(mid,   mid, k=6, s=1, p=5), nn.AvgPool2d(2, 2, 1),
            PDNBlock(mid,   mid, k=4, s=1, p=3), PDNBlock(mid, mid, k=4, s=1, p=3),
        )
        self.b3 = nn.Sequential(
            PDNBlock(in_ch, mid, k=8, s=1, p=7), nn.AvgPool2d(2, 2, 1),
            PDNBlock(mid,   mid, k=8, s=1, p=7), nn.AvgPool2d(2, 2, 1),
            PDNBlock(mid,   mid, k=4, s=1, p=3), PDNBlock(mid, mid, k=4, s=1, p=3),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(3 * mid, out_ch, 1),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True),
        )

    def forward(self, x):
        f1, f2, f3 = self.b1(x), self.b2(x), self.b3(x)
        h = min(f1.shape[2], f2.shape[2], f3.shape[2])
        w = min(f1.shape[3], f2.shape[3], f3.shape[3])
        return self.fuse(torch.cat(
            [f1[:,:,:h,:w], f2[:,:,:h,:w], f3[:,:,:h,:w]], dim=1))


# ── 8B. Teacher Backbone (ResNet-18, frozen) ───────────────────────────────
class TeacherBackbone(nn.Module):
    def __init__(self, proj_ch=384):
        super().__init__()
        rn = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
        self.stem   = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.layer3 = rn.layer3
        self.proj   = nn.Conv2d(256, proj_ch, 1)
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return self.proj(x)


# ── 8C. Reconstruction Autoencoder (skip connections) ─────────────────────
class ReconAE(nn.Module):
    def __init__(self, out_ch=384):
        super().__init__()
        def cbr(i, o, k=3, s=1, p=1):
            return nn.Sequential(nn.Conv2d(i,o,k,s,p), nn.BatchNorm2d(o),
                                 nn.LeakyReLU(0.1, True))
        self.enc1 = cbr(3,   64); self.pool1 = nn.AvgPool2d(2)
        self.enc2 = cbr(64, 128); self.pool2 = nn.AvgPool2d(2)
        self.enc3 = cbr(128,256); self.pool3 = nn.AvgPool2d(2)
        self.bot  = cbr(256, 256)
        self.up3  = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec3 = cbr(256, 128)
        self.up2  = nn.ConvTranspose2d(128,  64, 2, 2)
        self.dec2 = cbr(128,  64)
        self.up1  = nn.ConvTranspose2d(64,   64, 2, 2)
        self.out  = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x); p1 = self.pool1(e1)
        e2 = self.enc2(p1); p2 = self.pool2(e2)
        e3 = self.enc3(p2); p3 = self.pool3(e3)
        b  = self.bot(p3)
        d3 = self.dec3(torch.cat([self.up3(b),
             e2[:,:,:b.shape[2]*2,:b.shape[3]*2]], 1))
        d2 = self.dec2(torch.cat([self.up2(d3),
             e1[:,:,:d3.shape[2]*2,:d3.shape[3]*2]], 1))
        return self.out(self.up1(d2))


# ── Instantiate ────────────────────────────────────────────────────────────
teacher_net = TeacherBackbone(proj_ch=CFG.EAD_PDN_CH).to(DEVICE).eval()
student_net = MultiScalePDN(in_ch=3, out_ch=CFG.EAD_PDN_CH).to(DEVICE)
ae_net      = ReconAE(out_ch=CFG.EAD_PDN_CH).to(DEVICE)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"✅ Teacher (ResNet-18, frozen) : {sum(p.numel() for p in teacher_net.parameters()):,}")
print(f"✅ Student (MS-PDN, trainable) : {count_params(student_net):,}")
print(f"✅ AutoEncoder (trainable)     : {count_params(ae_net):,}")

with torch.no_grad():
    _x = torch.zeros(2, 3, 64, 64, device=DEVICE)
    _t = teacher_net(imagenet_norm(_x))
    _s = student_net(_x)
    _a = ae_net(_x)
    print(f"\n📐 Feature map shapes (batch=2, input 64×64):")
    print(f"   Teacher : {tuple(_t.shape)}")
    print(f"   Student : {tuple(_s.shape)}")
    print(f"   AutoEnc : {tuple(_a.shape)}")

✅ Active device: cuda
   GPU : Tesla T4
   Compute capability : sm_75
   PyTorch CUDA build : 12.8
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 124MB/s] 


✅ Teacher (ResNet-18, frozen) : 2,881,472
✅ Student (MS-PDN, trainable) : 3,671,168
✅ AutoEncoder (trainable)     : 1,536,960

📐 Feature map shapes (batch=2, input 64×64):
   Teacher : (2, 384, 4, 4)
   Student : (2, 384, 25, 25)
   AutoEnc : (2, 384, 64, 64)


In [9]:
# ── Cell 9: Hard-Feature Loss & EfficientAD Training Loop ──────────────────

class HardFeatureLoss(nn.Module):
    def __init__(self, hard_frac=0.2, ood_weight=0.1):
        super().__init__()
        self.hard_frac  = hard_frac
        self.ood_weight = ood_weight

    def forward(self, t_feat, s_feat, ae_feat, x_normal):
        h = min(t_feat.shape[2], s_feat.shape[2])
        w = min(t_feat.shape[3], s_feat.shape[3])
        t_feat  = t_feat[:, :, :h, :w]
        s_feat  = s_feat[:, :, :h, :w]
        ae_feat = F.interpolate(ae_feat, size=(h, w),
                                mode='bilinear', align_corners=False)

        sq_diff = ((t_feat - s_feat) ** 2).mean(dim=1)   # (B, h, w)

        B, hh, ww = sq_diff.shape
        flat   = sq_diff.view(B, -1)
        k      = max(1, int(hh * ww * self.hard_frac))
        thresh = flat.topk(k, dim=1).values[:, -1].unsqueeze(1)
        hard_mask    = (flat >= thresh).float()
        student_loss = (flat * hard_mask).sum() / (hard_mask.sum() + 1e-8)

        ae_loss = F.mse_loss(ae_feat, t_feat.detach())

        idx   = torch.randperm(x_normal.shape[1], device=x_normal.device)
        x_ood = x_normal[:, idx]
        s_ood = student_net(x_ood)[:, :, :h, :w]
        ood_loss = 1.0 / (s_ood.pow(2).mean() + 1e-8)

        total = student_loss + ae_loss + self.ood_weight * ood_loss
        return total, student_loss.item(), ae_loss.item()


ead_criterion = HardFeatureLoss(hard_frac=0.2, ood_weight=0.1)
ead_optimizer = torch.optim.AdamW(
    list(student_net.parameters()) + list(ae_net.parameters()),
    lr=CFG.EAD_LR, weight_decay=CFG.EAD_WD,
)
ead_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    ead_optimizer, T_max=CFG.EAD_EPOCHS, eta_min=1e-6,
)

EAD_HISTORY = {'loss': [], 'student_loss': [], 'ae_loss': []}
teacher_net.eval()

print(f"🔄 Training EfficientAD for {CFG.EAD_EPOCHS} epochs …")
print(f"   Target training loss : {CFG.EAD_TARGET_LOSS}")
print("-" * 60)

for epoch in range(1, CFG.EAD_EPOCHS + 1):
    student_net.train(); ae_net.train()
    ep_loss = ep_s = ep_a = 0.0

    for (x_batch,) in train_loader:
        x_batch = x_batch.to(DEVICE)
        with torch.no_grad():
            t_feat = teacher_net(imagenet_norm(x_batch))
        s_feat  = student_net(x_batch)
        ae_feat = ae_net(x_batch)

        loss, sl, al = ead_criterion(t_feat, s_feat, ae_feat, x_batch)

        ead_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(student_net.parameters()) + list(ae_net.parameters()), 1.0)
        ead_optimizer.step()

        ep_loss += loss.item(); ep_s += sl; ep_a += al

    n = len(train_loader)
    EAD_HISTORY['loss'].append(ep_loss / n)
    EAD_HISTORY['student_loss'].append(ep_s / n)
    EAD_HISTORY['ae_loss'].append(ep_a / n)
    ead_scheduler.step()

    if epoch % 50 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>3}/{CFG.EAD_EPOCHS} | "
              f"Total={ep_loss/n:.5f} | Student={ep_s/n:.5f} | AE={ep_a/n:.5f}")

print(f"\n✅ EfficientAD training complete.")
print(f"   Final loss: {EAD_HISTORY['loss'][-1]:.5f}  (target: {CFG.EAD_TARGET_LOSS})")


🔄 Training EfficientAD for 200 epochs …
   Target training loss : 0.01564
------------------------------------------------------------
  Epoch   1/200 | Total=0.52554 | Student=0.18394 | AE=0.04803
  Epoch  50/200 | Total=0.25770 | Student=0.07041 | AE=0.00224
  Epoch 100/200 | Total=0.17040 | Student=0.05212 | AE=0.00131
  Epoch 150/200 | Total=0.21694 | Student=0.05337 | AE=0.00115
  Epoch 200/200 | Total=0.16382 | Student=0.05621 | AE=0.00112

✅ EfficientAD training complete.
   Final loss: 0.16382  (target: 0.01564)


In [10]:
# ── Cell 10: EfficientAD Training Curves ───────────────────────────────────
apply_pub_style()

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor='white')
fig.subplots_adjust(wspace=0.32)

epochs = range(1, len(EAD_HISTORY['loss']) + 1)
labels = ['Total Loss', 'Student Loss', 'AE Loss']
keys   = ['loss', 'student_loss', 'ae_loss']
colors = [CFG.C_EAD, CFG.C_DDPM, CFG.C_FUSION]

for ax, key, label, color in zip(axes, keys, labels, colors):
    raw = np.array(EAD_HISTORY[key])
    ax.plot(epochs, raw, color=color, linewidth=1.5, alpha=0.4, label='Raw')
    if len(epochs) > 10:
        smooth = uniform_filter1d(raw, size=max(1, len(epochs)//10))
        ax.plot(epochs, smooth, color=color, linewidth=2.2, label='Smoothed')
    ax.axhline(raw[-1], color='#555555', linewidth=0.9, linestyle=':',
               label=f'Final = {raw[-1]:.5f}')
    ax.set_title(f'EfficientAD — {label}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)

fig.suptitle('EfficientAD Training Curves (Structural Module)',
             fontsize=13, fontweight='bold', y=1.02)
fig.savefig(f'{CFG.OUT_DIR}/02_ead_training_curves.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 02_ead_training_curves.png")


✅ Figure saved → 02_ead_training_curves.png


In [12]:
# ── Cell 11: EfficientAD Anomaly Scoring ───────────────────────────────────
student_net.eval(); ae_net.eval()

def compute_ead_anomaly_scores(loader, device=DEVICE):
    all_maps, all_scores = [], []
    with torch.no_grad():
        for (x,) in loader:
            x = x.to(device)
            t_feat = teacher_net(imagenet_norm(x))
            s_feat = student_net(x)
            h = min(t_feat.shape[2], s_feat.shape[2])
            w = min(t_feat.shape[3], s_feat.shape[3])
            diff_map = ((t_feat[:,:,:h,:w] - s_feat[:,:,:h,:w])**2).mean(1, keepdim=True)
            diff_up  = F.interpolate(diff_map, size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
                                     mode='bilinear', align_corners=False).squeeze(1)
            diff_up_np = diff_up.cpu().numpy()
            all_maps.append(diff_up_np)
            all_scores.append(diff_up_np.reshape(len(x), -1).mean(axis=1))
    return np.concatenate(all_maps), np.concatenate(all_scores)

print("🔄 Computing EfficientAD anomaly scores (val set) …")
_, ead_val_scores_raw = compute_ead_anomaly_scores(val_loader)

val_normal_mask = (y_val == 0)
q_lo  = np.percentile(ead_val_scores_raw[val_normal_mask], 1)
q_hi  = np.percentile(ead_val_scores_raw[~val_normal_mask]
                       if (~val_normal_mask).any()
                       else ead_val_scores_raw, 99)

def normalise_ead(scores):
    return np.clip((scores - q_lo) / (q_hi - q_lo + 1e-8), 0.0, 1.0)

ead_val_scores = normalise_ead(ead_val_scores_raw)
fpr_v, tpr_v, thr_v = roc_curve(y_val, ead_val_scores)
opt_thr_ead = thr_v[np.argmax(tpr_v - fpr_v)]

print("🔄 Computing EfficientAD anomaly scores (test set) …")
ead_test_maps, ead_test_scores_raw = compute_ead_anomaly_scores(test_loader)
ead_test_scores = normalise_ead(ead_test_scores_raw)

ead_auroc = roc_auc_score(y_test, ead_test_scores)
ead_ap    = average_precision_score(y_test, ead_test_scores)
ead_pred  = (ead_test_scores >= opt_thr_ead).astype(int)
ead_f1    = f1_score(y_test, ead_pred)
ead_mcc   = matthews_corrcoef(y_test, ead_pred)

print(f"\n📊 EfficientAD Test Results:")
print(f"   AUROC         : {ead_auroc:.4f}  (target: 0.9987)")
print(f"   Avg Precision : {ead_ap:.4f}")
print(f"   F1            : {ead_f1:.4f}")
print(f"   MCC           : {ead_mcc:.4f}")
print(f"   Threshold τ   : {opt_thr_ead:.4f}")
print(f"   Final Loss    : {EAD_HISTORY['loss'][-1]:.5f}")


🔄 Computing EfficientAD anomaly scores (val set) …
🔄 Computing EfficientAD anomaly scores (test set) …

📊 EfficientAD Test Results:
   AUROC         : 0.9834  (target: 0.9987)
   Avg Precision : 0.9998
   F1            : 0.9692
   MCC           : 0.3116
   Threshold τ   : 0.2201
   Final Loss    : 0.16382


In [13]:
# ── Cell 12: EfficientAD Results Visualisation ─────────────────────────────
apply_pub_style()

fig = plt.figure(figsize=(20, 9), facecolor='white')
gs  = gridspec.GridSpec(2, 5, figure=fig, hspace=0.50, wspace=0.35)

# ROC
ax_roc = fig.add_subplot(gs[0, :2])
fpr_t, tpr_t, _ = roc_curve(y_test, ead_test_scores)
ax_roc.plot(fpr_t, tpr_t, color=CFG.C_EAD, linewidth=2.2,
            label=f'EfficientAD  AUROC = {ead_auroc:.4f}')
ax_roc.plot([0,1],[0,1], color='#aaaaaa', linewidth=1.0, linestyle='--', label='Random')
ax_roc.fill_between(fpr_t, tpr_t, alpha=0.08, color=CFG.C_EAD)
ax_roc.set_title('ROC Curve — EfficientAD (Test Set)', fontweight='bold')
ax_roc.set_xlabel('False Positive Rate'); ax_roc.set_ylabel('True Positive Rate')
ax_roc.legend()
ax_roc.set_xlim(0, 1); ax_roc.set_ylim(0, 1.02)

# Score Distribution
ax_sd = fig.add_subplot(gs[0, 2:4])
ax_sd.hist(ead_test_scores[y_test==0], bins=50, color=CFG.C_NORMAL,
           alpha=0.75, density=True, label='Normal', edgecolor='white', linewidth=0.3)
ax_sd.hist(ead_test_scores[y_test==1], bins=50, color=CFG.C_DEFECT,
           alpha=0.75, density=True, label='Defective', edgecolor='white', linewidth=0.3)
ax_sd.axvline(opt_thr_ead, color='#333333', linewidth=1.5, linestyle='--',
              label=f'τ = {opt_thr_ead:.3f}')
ax_sd.set_title('Score Distribution', fontweight='bold')
ax_sd.set_xlabel('Normalised Anomaly Score'); ax_sd.set_ylabel('Density')
ax_sd.legend()

# PR Curve
ax_pr = fig.add_subplot(gs[0, 4])
prec, rec, _ = precision_recall_curve(y_test, ead_test_scores)
ax_pr.plot(rec, prec, color=CFG.C_BASE, linewidth=2.0, label=f'AP = {ead_ap:.4f}')
ax_pr.fill_between(rec, prec, alpha=0.08, color=CFG.C_BASE)
ax_pr.set_title('PR Curve', fontweight='bold')
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.legend()

# Sample Anomaly Maps
n_show = 5
test_arr = T_test.numpy()
sample_idxs = np.linspace(0, len(y_test)-1, n_show, dtype=int)

for col, si in enumerate(sample_idxs):
    raw_img  = (test_arr[si, 2] + 1) / 2
    anom_map = ead_test_maps[si]
    anom_n   = (anom_map - anom_map.min()) / (anom_map.max() - anom_map.min() + 1e-8)
    label_s  = 'DEF' if y_test[si] else 'NOR'
    pred_s   = 'DEF' if ead_pred[si] else 'NOR'
    correct  = '✓' if label_s == pred_s else '✗'
    border_c = CFG.C_FUSION if label_s == pred_s else CFG.C_DDPM

    ax_i = fig.add_subplot(gs[1, col])
    # Show wafer map with anomaly map overlay
    ax_i.imshow(raw_img, cmap='gray', interpolation='nearest', alpha=0.55)
    ax_i.imshow(anom_n, cmap=ANOM_CMAP, interpolation='nearest', alpha=0.60)
    ax_i.set_title(f'{correct} GT:{label_s}\nPred:{pred_s}',
                   fontsize=9, fontweight='bold',
                   color=CFG.C_FUSION if label_s == pred_s else CFG.C_DDPM)
    ax_i.axis('off')
    for spine in ax_i.spines.values():
        spine.set_visible(True); spine.set_edgecolor(border_c); spine.set_linewidth(2)

fig.suptitle('EfficientAD — Structural Module Results',
             fontsize=13, fontweight='bold', y=1.01)
fig.savefig(f'{CFG.OUT_DIR}/03_ead_results.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 03_ead_results.png")


✅ Figure saved → 03_ead_results.png


---
# Part II — DDPM: Generative Reconstruction Module

**Role:** detect *logical* anomalies — invalid spatial combinations of otherwise
normal-looking features — by learning the full distribution of defect-free wafer maps.

| Published Result | Value |
|-----------------|-------|
| U-Net Parameters | 7.16 M |
| Training Loss | 0.01633 |
| Epochs | 30 |
| Sampling Rate (DDIM) | 45.34 it/s |


In [14]:
# ── Cell 13: Diffusion Utilities: Cosine Schedule & EMA ───────────────────

def cosine_beta_schedule(T: int, s: float = 0.008):
    steps    = T + 1
    t        = torch.linspace(0, T, steps, dtype=torch.float64)
    f_t      = torch.cos((t / T + s) / (1 + s) * math.pi / 2) ** 2
    alpha_bar = f_t / f_t[0]
    beta     = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return torch.clip(beta, 0.0001, 0.9999).float()

T_total        = CFG.DDPM_T
beta           = cosine_beta_schedule(T_total).to(DEVICE)
alpha          = 1.0 - beta
alpha_bar      = torch.cumprod(alpha, dim=0)
alpha_bar_prev = F.pad(alpha_bar[:-1], (1, 0), value=1.0)
sqrt_ab        = alpha_bar.sqrt()
sqrt_one_m_ab  = (1.0 - alpha_bar).sqrt()

def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    _sqrt_ab      = sqrt_ab[t].view(-1,1,1,1)
    _sqrt_one_m   = sqrt_one_m_ab[t].view(-1,1,1,1)
    return _sqrt_ab * x0 + _sqrt_one_m * noise

print(f"✅ Cosine noise schedule ready:")
print(f"   T={T_total} | β range [{beta[0]:.5f}, {beta[-1]:.5f}]")
print(f"   ᾱ at t=500 : {alpha_bar[500]:.4f}")


class EMAHelper:
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay  = decay
        self.shadow = {k: v.clone().detach() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for k, v in model.state_dict().items():
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v

    def apply_shadow(self, model: nn.Module):
        model.load_state_dict(self.shadow)

print("✅ EMAHelper class defined (decay=0.9999)")


✅ Cosine noise schedule ready:
   T=1000 | β range [0.00010, 0.99990]
   ᾱ at t=500 : 0.4921
✅ EMAHelper class defined (decay=0.9999)


In [17]:
# ── Cell 14: U-Net Architecture (~7.16 M Parameters) ──────────────────────

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__(); self.dim = dim
    def forward(self, t):
        device = t.device; half = self.dim // 2
        freqs = torch.exp(-math.log(10000) *
                          torch.arange(half, device=device) / (half - 1))
        args = t[:, None].float() * freqs[None]
        return torch.cat([args.sin(), args.cos()], dim=-1)


class TimeEmbedding(nn.Module):
    def __init__(self, sin_dim: int, time_dim: int):
        super().__init__()
        self.sin_emb = SinusoidalPositionEmbedding(sin_dim)
        self.mlp = nn.Sequential(
            nn.Linear(sin_dim, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
    def forward(self, t): return self.mlp(self.sin_emb(t))


def _gn(channels: int) -> nn.GroupNorm:
    """
    GroupNorm with a safe group count.
    Picks the largest power-of-2 divisor of `channels` that is <= 32.
    This is deterministic, depends only on `channels`, and always divides evenly.
    """
    g = 32
    while g > 1 and channels % g != 0:
        g //= 2
    return nn.GroupNorm(g, channels)


class ResBlock(nn.Module):
    """
    ResBlock where norm group counts are computed from the channel argument,
    not from a shared mutable variable — prevents all GroupNorm shape mismatches.
    """
    def __init__(self, in_ch: int, out_ch: int, time_dim: int):
        super().__init__()
        self.norm1     = _gn(in_ch)          # norm on in_ch
        self.conv1     = nn.Conv2d(in_ch,  out_ch, 3, 1, 1)
        self.norm2     = _gn(out_ch)         # norm on out_ch
        self.conv2     = nn.Conv2d(out_ch, out_ch, 3, 1, 1)
        self.act       = nn.SiLU()
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.skip      = (nn.Conv2d(in_ch, out_ch, 1)
                          if in_ch != out_ch else nn.Identity())

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = h + self.time_proj(self.act(t_emb))[:, :, None, None]
        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)


class SelfAttentionBlock(nn.Module):
    def __init__(self, channels: int, heads: int = 4):
        super().__init__()
        self.norm = _gn(channels)
        self.attn = nn.MultiheadAttention(channels, heads, batch_first=True)

    def forward(self, x: torch.Tensor):
        B, C, H, W = x.shape
        h = self.norm(x).view(B, C, -1).permute(0, 2, 1)
        h, _ = self.attn(h, h, h)
        return x + h.permute(0, 2, 1).view(B, C, H, W)


class UNet(nn.Module):
    """
    U-Net ε-predictor.

    Channel layout (base_ch=48, mults=(1,2,4)):
        Encoder : 48 → 96 → 192
        Bottleneck: 192 → 192 (3 ResBlocks + attention)
        Decoder : 192+192→192 → 96+96→96 → 48+48→48
        Output  : Conv2d(48, 3)

    Parameter count ~7.16 M achieved by 3 bottleneck ResBlocks instead of 2.
    """
    def __init__(
        self,
        in_ch    : int   = 3,
        out_ch   : int   = 3,
        base_ch  : int   = 48,
        ch_mults : tuple = (1, 2, 4),   # → [48, 96, 192]
        n_blocks : int   = 2,           # ResBlocks per encoder/decoder level
        n_mid    : int   = 3,           # ResBlocks in bottleneck (key for param count)
        time_dim : int   = 320,
        attn_res : tuple = (16,),
    ):
        super().__init__()

        # Fully explicit channel list — no mutable tracking variables
        C = [base_ch * m for m in ch_mults]   # [48, 96, 192]
        n_levels = len(C)

        self.time_emb  = TimeEmbedding(sin_dim=128, time_dim=time_dim)
        self.init_conv = nn.Conv2d(in_ch, C[0], 3, 1, 1)

        # ── Encoder ──────────────────────────────────────────────────────────
        # enc_in_ch[lv][bi] = exact in_channels for ResBlock (lv, bi)
        #   bi=0 : input comes from previous level output (or init_conv for lv=0)
        #   bi>0 : input comes from previous ResBlock in same level → out is C[lv]
        self.enc_blocks = nn.ModuleList()
        self.down_ops   = nn.ModuleList()
        cur_res = CFG.IMG_SIZE   # 64

        for lv in range(n_levels):
            in_ch_first = C[lv - 1] if lv > 0 else C[0]   # after init_conv C[0]
            # NOTE: down_op changes channels from C[lv-1] → C[lv], so the first
            #       ResBlock in lv>0 receives C[lv] channels (down_op is strided conv
            #       with out_channels=C[lv]), NOT C[lv-1].
            # The strided conv is defined below for lv < n_levels-1 with out=C[lv+1],
            # so the first block at lv receives C[lv] from the previous down_op.
            level = nn.ModuleList()
            for bi in range(n_blocks):
                blk_in = C[lv] if bi > 0 else (C[0] if lv == 0 else C[lv])
                level.append(ResBlock(blk_in, C[lv], time_dim))
                if cur_res in attn_res:
                    level.append(SelfAttentionBlock(C[lv]))
            self.enc_blocks.append(level)

            if lv < n_levels - 1:
                # strided conv: C[lv] → C[lv+1], halves spatial resolution
                self.down_ops.append(
                    nn.Conv2d(C[lv], C[lv + 1], 3, stride=2, padding=1))
                cur_res //= 2
            else:
                self.down_ops.append(nn.Identity())

        # ── Bottleneck ───────────────────────────────────────────────────────
        bot = C[-1]   # 192
        mid_layers = []
        mid_layers.append(ResBlock(bot, bot, time_dim))
        mid_layers.append(SelfAttentionBlock(bot))
        for _ in range(n_mid - 1):          # 2 extra ResBlocks → 3 total
            mid_layers.append(ResBlock(bot, bot, time_dim))
        self.mid = nn.ModuleList(mid_layers)

        # ── Decoder ──────────────────────────────────────────────────────────
        # At decoder level lv:
        #   skip from encoder level (n_levels-1-lv) has C[n_levels-1-lv] channels
        #   upsampled tensor has C[n_levels-1-lv] channels (or C[-1] for lv=0)
        #   → concat → in_ch = 2 * C[n_levels-1-lv]
        #   → out_ch = C[n_levels-1-lv]
        # up_op (ConvTranspose2d) then maps C[enc_lv] → C[enc_lv - 1]
        self.dec_blocks = nn.ModuleList()
        self.up_ops     = nn.ModuleList()
        cur_res = CFG.IMG_SIZE // (2 ** (n_levels - 1))   # 16

        for lv in range(n_levels):
            enc_lv  = n_levels - 1 - lv          # 2, 1, 0
            skip_ch = C[enc_lv]                   # 192, 96, 48
            up_ch   = C[enc_lv]                   # tensor coming in from up_op
            out_ch_ = C[enc_lv]                   # output of this decoder level

            level = nn.ModuleList()
            for bi in range(n_blocks):
                blk_in = (up_ch + skip_ch) if bi == 0 else out_ch_
                level.append(ResBlock(blk_in, out_ch_, time_dim))
                if cur_res in attn_res:
                    level.append(SelfAttentionBlock(out_ch_))
            self.dec_blocks.append(level)

            if lv < n_levels - 1:
                next_enc_lv = enc_lv - 1         # 1, 0
                self.up_ops.append(
                    nn.ConvTranspose2d(out_ch_, C[next_enc_lv], 2, stride=2))
                cur_res *= 2
            else:
                self.up_ops.append(nn.Identity())

        # ── Output head ──────────────────────────────────────────────────────
        self.final_norm = _gn(C[0])
        self.final_conv = nn.Conv2d(C[0], out_ch, 3, 1, 1)

    # ── Forward helpers ──────────────────────────────────────────────────────
    def _enc_level(self, x, t_emb, blocks):
        for blk in blocks:
            x = blk(x) if isinstance(blk, SelfAttentionBlock) else blk(x, t_emb)
        return x

    def _dec_level(self, x, t_emb, skip, blocks):
        x = torch.cat([x, skip], dim=1)
        for blk in blocks:
            x = blk(x) if isinstance(blk, SelfAttentionBlock) else blk(x, t_emb)
        return x

    def forward(self, x: torch.Tensor, t: torch.Tensor):
        t_emb = self.time_emb(t)
        x     = self.init_conv(x)            # (B, 48, 64, 64)

        # Encoder
        skips = []
        for level_blocks, down in zip(self.enc_blocks, self.down_ops):
            x = self._enc_level(x, t_emb, level_blocks)
            skips.append(x)
            x = down(x)

        # Bottleneck
        for blk in self.mid:
            x = blk(x) if isinstance(blk, SelfAttentionBlock) else blk(x, t_emb)

        # Decoder
        for lv, (level_blocks, up) in enumerate(zip(self.dec_blocks, self.up_ops)):
            x = self._dec_level(x, t_emb, skips[-(lv + 1)], level_blocks)
            x = up(x)

        return self.final_conv(F.silu(self.final_norm(x)))


# ── Instantiate & verify ──────────────────────────────────────────────────────
ddpm_unet = UNet().to(DEVICE)
n_params  = sum(p.numel() for p in ddpm_unet.parameters())
print(f"✅ DDPM U-Net instantiated")
print(f"   Parameters : {n_params/1e6:.2f} M  (target: {CFG.DDPM_TARGET_PARAMS} M)")
print(f"   Channels   : {[48 * m for m in CFG.DDPM_CH_MULTS]}")
print(f"   Time dim   : {CFG.DDPM_TIME_DIM}")
print(f"   Attention  : at resolutions {CFG.DDPM_ATTN_RES}")
print(f"   Bottleneck : 3 ResBlocks + attention")

with torch.no_grad():
    _x   = torch.zeros(2, 3, 64, 64, device=DEVICE)
    _t   = torch.randint(0, T_total, (2,), device=DEVICE)
    _out = ddpm_unet(_x, _t)
    print(f"\n📐 I/O: Input {tuple(_x.shape)} → Output {tuple(_out.shape)}")
    assert _out.shape == _x.shape, f"Shape mismatch: {_out.shape} != {_x.shape}"
    print("✅ Shape check passed.")

✅ DDPM U-Net instantiated
   Parameters : 7.83 M  (target: 7.16 M)
   Channels   : [48, 96, 192]
   Time dim   : 320
   Attention  : at resolutions (16,)
   Bottleneck : 3 ResBlocks + attention

📐 I/O: Input (2, 3, 64, 64) → Output (2, 3, 64, 64)
✅ Shape check passed.


In [18]:
# ── Cell 15: DDPM Training (30 Epochs, Cosine Schedule, EMA) ───────────────
ddpm_optimizer = torch.optim.AdamW(
    ddpm_unet.parameters(), lr=CFG.DDPM_LR, weight_decay=1e-5)
ddpm_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    ddpm_optimizer, T_max=CFG.DDPM_EPOCHS, eta_min=1e-6)
ddpm_ema = EMAHelper(ddpm_unet, decay=CFG.DDPM_EMA_DECAY)

ddpm_loader = DataLoader(TensorDataset(T_train), batch_size=CFG.DDPM_BS,
                         shuffle=True, num_workers=_n_workers,
                         pin_memory=True, drop_last=True)

DDPM_HISTORY = {'loss': []}

print(f"🔄 Training DDPM for {CFG.DDPM_EPOCHS} epochs …")
print(f"   U-Net params   : {n_params/1e6:.2f} M  (target {CFG.DDPM_TARGET_PARAMS} M)")
print(f"   Target loss    : {CFG.DDPM_TARGET_LOSS}")
print(f"   Batch size     : {CFG.DDPM_BS}")
print("-" * 60)

for epoch in range(1, CFG.DDPM_EPOCHS + 1):
    ddpm_unet.train()
    ep_loss = 0.0
    for (x0,) in ddpm_loader:
        x0  = x0.to(DEVICE)
        t   = torch.randint(0, T_total, (x0.shape[0],), device=DEVICE)
        eps = torch.randn_like(x0)
        x_t = q_sample(x0, t, noise=eps)
        eps_pred = ddpm_unet(x_t, t)
        loss = F.mse_loss(eps_pred, eps)
        ddpm_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ddpm_unet.parameters(), 1.0)
        ddpm_optimizer.step()
        ddpm_ema.update(ddpm_unet)
        ep_loss += loss.item()

    ep_loss /= len(ddpm_loader)
    DDPM_HISTORY['loss'].append(ep_loss)
    ddpm_scheduler.step()
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>3}/{CFG.DDPM_EPOCHS} | Loss = {ep_loss:.5f}")

_backup = copy.deepcopy(ddpm_unet.state_dict())
ddpm_ema.apply_shadow(ddpm_unet)
ddpm_unet.eval()

print(f"\n✅ DDPM training complete.")
print(f"   Final loss   : {DDPM_HISTORY['loss'][-1]:.5f}  (target: {CFG.DDPM_TARGET_LOSS})")
print(f"   EMA weights applied for inference.")


🔄 Training DDPM for 30 epochs …
   U-Net params   : 7.83 M  (target 7.16 M)
   Target loss    : 0.01633
   Batch size     : 32
------------------------------------------------------------
  Epoch   1/30 | Loss = 0.65691
  Epoch   5/30 | Loss = 0.14093
  Epoch  10/30 | Loss = 0.08556
  Epoch  15/30 | Loss = 0.07160
  Epoch  20/30 | Loss = 0.05481
  Epoch  25/30 | Loss = 0.05400
  Epoch  30/30 | Loss = 0.04879

✅ DDPM training complete.
   Final loss   : 0.04879  (target: 0.01633)
   EMA weights applied for inference.


In [19]:
# ── Cell 16: DDIM Sampler & Sampling Speed Benchmark ──────────────────────

@torch.no_grad()
def ddim_sample(model, shape, steps=50, device=DEVICE, eta=0.0):
    t_seq = torch.linspace(T_total - 1, 0, steps + 1).long().to(device)
    x     = torch.randn(shape, device=device)
    for i in range(len(t_seq) - 1):
        t_cur = t_seq[i]; t_nxt = t_seq[i + 1]
        eps   = model(x, t_cur.expand(shape[0]))
        ab_t  = alpha_bar[t_cur]
        ab_n  = alpha_bar[t_nxt] if t_nxt >= 0 else torch.tensor(1.0, device=device)
        x0_p  = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1.0, 1.0)
        sigma = eta * ((1-ab_n)/(1-ab_t) * (1-ab_t/ab_n)).sqrt()
        noise = torch.randn_like(x) if eta > 0 else 0
        x     = ab_n.sqrt() * x0_p + (1-ab_n-sigma**2).sqrt() * eps + sigma * noise
    return x.clamp(-1.0, 1.0)

@torch.no_grad()
def ddim_denoise_from(model, x_t, t_start, steps=50, device=DEVICE):
    t_seq = torch.linspace(t_start, 0, steps + 1).long().to(device)
    x = x_t.clone()
    for i in range(len(t_seq) - 1):
        t_cur = t_seq[i]; t_nxt = t_seq[i + 1]
        eps   = model(x, t_cur.expand(x.shape[0]))
        ab_t  = alpha_bar[t_cur]
        ab_n  = alpha_bar[t_nxt] if t_nxt > 0 else torch.tensor(1.0, device=device)
        x0_p  = ((x - (1-ab_t).sqrt()*eps) / ab_t.sqrt()).clamp(-1.0, 1.0)
        x     = ab_n.sqrt() * x0_p + (1-ab_n).sqrt() * eps
    return x.clamp(-1.0, 1.0)

print(f"⏱  Benchmarking DDIM sampling ({CFG.DDPM_DDIM_STEPS} steps) …")
_shape = (CFG.DDPM_BS, 3, CFG.IMG_SIZE, CFG.IMG_SIZE)
ddim_sample(ddpm_unet, _shape, steps=CFG.DDPM_DDIM_STEPS)   # warm-up

N_BENCH = 5
t0 = time.perf_counter()
for _ in range(N_BENCH):
    ddim_sample(ddpm_unet, _shape, steps=CFG.DDPM_DDIM_STEPS)
elapsed = time.perf_counter() - t0
sampling_rate = N_BENCH * CFG.DDPM_DDIM_STEPS / elapsed
print(f"✅ Sampling rate : {sampling_rate:.2f} it/s  (target: 45.34 it/s)")


⏱  Benchmarking DDIM sampling (50 steps) …
✅ Sampling rate : 13.22 it/s  (target: 45.34 it/s)


In [20]:
# ── Cell 17: DDPM Reconstruction-Based Anomaly Scoring ─────────────────────
RECON_T = CFG.DDPM_RECON_T   # 500

def compute_ddpm_anomaly_scores(loader, t_star=RECON_T, ddim_steps=50, device=DEVICE):
    all_maps, all_scores = [], []
    with torch.no_grad():
        for (x,) in loader:
            x = x.to(device)
            B = x.shape[0]
            t_tensor = torch.full((B,), t_star, device=device, dtype=torch.long)
            x_noisy  = q_sample(x, t_tensor)
            x_recon  = ddim_denoise_from(ddpm_unet, x_noisy, t_start=t_star, steps=ddim_steps)
            amap     = ((x - x_recon)**2).mean(dim=1)    # (B, 64, 64)
            amap_np  = amap.cpu().numpy()
            all_maps.append(amap_np)
            all_scores.append(amap_np.reshape(B, -1).mean(axis=1))
    return np.concatenate(all_maps), np.concatenate(all_scores)

print(f"🔄 Computing DDPM reconstruction scores (val set, t*={RECON_T}) …")
ddpm_val_maps, ddpm_val_scores_raw = compute_ddpm_anomaly_scores(val_loader)

val_norm_mask = (y_val == 0)
q_lo_d = np.percentile(ddpm_val_scores_raw[val_norm_mask], 1)
q_hi_d = np.percentile(ddpm_val_scores_raw[~val_norm_mask]
                        if (~val_norm_mask).any() else ddpm_val_scores_raw, 99)

def normalise_ddpm(sc):
    return np.clip((sc - q_lo_d) / (q_hi_d - q_lo_d + 1e-8), 0.0, 1.0)

ddpm_val_scores = normalise_ddpm(ddpm_val_scores_raw)
fpr_dv, tpr_dv, thr_dv = roc_curve(y_val, ddpm_val_scores)
opt_thr_ddpm = thr_dv[np.argmax(tpr_dv - fpr_dv)]

print("🔄 Computing DDPM reconstruction scores (test set) …")
ddpm_test_maps, ddpm_test_scores_raw = compute_ddpm_anomaly_scores(test_loader)
ddpm_test_scores = normalise_ddpm(ddpm_test_scores_raw)

ddpm_auroc = roc_auc_score(y_test, ddpm_test_scores)
ddpm_ap    = average_precision_score(y_test, ddpm_test_scores)
ddpm_pred  = (ddpm_test_scores >= opt_thr_ddpm).astype(int)
ddpm_f1    = f1_score(y_test, ddpm_pred)
ddpm_mcc   = matthews_corrcoef(y_test, ddpm_pred)

print(f"\n📊 DDPM Test Results:")
print(f"   AUROC         : {ddpm_auroc:.4f}")
print(f"   Avg Precision : {ddpm_ap:.4f}")
print(f"   F1            : {ddpm_f1:.4f}")
print(f"   MCC           : {ddpm_mcc:.4f}")
print(f"   Final Loss    : {DDPM_HISTORY['loss'][-1]:.5f}  (target: {CFG.DDPM_TARGET_LOSS})")
print(f"   U-Net Params  : {n_params/1e6:.2f} M  (target: {CFG.DDPM_TARGET_PARAMS} M)")


🔄 Computing DDPM reconstruction scores (val set, t*=500) …
🔄 Computing DDPM reconstruction scores (test set) …

📊 DDPM Test Results:
   AUROC         : 0.6411
   Avg Precision : 0.9948
   F1            : 0.7096
   MCC           : 0.0332
   Final Loss    : 0.04879  (target: 0.01633)
   U-Net Params  : 7.83 M  (target: 7.16 M)


In [21]:
# ── Cell 18: DDPM Results Visualisation ────────────────────────────────────
apply_pub_style()

fig = plt.figure(figsize=(20, 11), facecolor='white')
gs  = gridspec.GridSpec(3, 5, figure=fig, hspace=0.50, wspace=0.35)

# Training Curve
ax_lc = fig.add_subplot(gs[0, :2])
eps_r = range(1, len(DDPM_HISTORY['loss']) + 1)
raw_d = np.array(DDPM_HISTORY['loss'])
ax_lc.plot(eps_r, raw_d, color=CFG.C_DDPM, linewidth=1.5, alpha=0.4, label='Raw loss')
ax_lc.plot(eps_r, uniform_filter1d(raw_d, size=3), color=CFG.C_DDPM,
           linewidth=2.2, label='Smoothed')
ax_lc.axhline(CFG.DDPM_TARGET_LOSS, color='#333333', linewidth=1.2, linestyle='--',
              label=f'Target {CFG.DDPM_TARGET_LOSS}')
ax_lc.set_title('DDPM Training Loss (30 Epochs)', fontweight='bold')
ax_lc.set_xlabel('Epoch'); ax_lc.set_ylabel('MSE Loss')
ax_lc.legend()

# ROC
ax_roc = fig.add_subplot(gs[0, 2:4])
fpr_d, tpr_d, _ = roc_curve(y_test, ddpm_test_scores)
ax_roc.plot(fpr_d, tpr_d, color=CFG.C_DDPM, linewidth=2.2,
            label=f'DDPM  AUROC = {ddpm_auroc:.4f}')
ax_roc.fill_between(fpr_d, tpr_d, alpha=0.08, color=CFG.C_DDPM)
ax_roc.plot([0,1],[0,1], color='#aaaaaa', linewidth=1.0, linestyle='--')
ax_roc.set_title('DDPM — ROC Curve (Test Set)', fontweight='bold')
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.legend(); ax_roc.set_xlim(0, 1); ax_roc.set_ylim(0, 1.02)

# Score Dist
ax_sd = fig.add_subplot(gs[0, 4])
ax_sd.hist(ddpm_test_scores[y_test==0], bins=40, color=CFG.C_NORMAL,
           alpha=0.75, density=True, label='Normal', edgecolor='white', linewidth=0.3)
ax_sd.hist(ddpm_test_scores[y_test==1], bins=40, color=CFG.C_DEFECT,
           alpha=0.75, density=True, label='Defective', edgecolor='white', linewidth=0.3)
ax_sd.axvline(opt_thr_ddpm, color='#333333', linewidth=1.5, linestyle='--',
              label=f'τ={opt_thr_ddpm:.3f}')
ax_sd.set_title('Score Distribution', fontweight='bold')
ax_sd.legend(fontsize=8)

# Sample Reconstructions
sids = np.linspace(0, len(y_test)-1, 5, dtype=int)
for col, si in enumerate(sids):
    orig_arr  = T_test[si].numpy()
    disp_orig = (orig_arr[2] + 1) / 2
    amap      = ddpm_test_maps[si]
    amap_n    = (amap - amap.min()) / (amap.max() - amap.min() + 1e-8)
    label_s   = 'DEF' if y_test[si] else 'NOR'

    ax_o = fig.add_subplot(gs[1, col])
    ax_o.imshow(disp_orig, cmap=WBM_CMAP, interpolation='nearest')
    ax_o.set_title(f'GT: {label_s}', fontsize=9, fontweight='bold')
    ax_o.axis('off')

    ax_m = fig.add_subplot(gs[2, col])
    ax_m.imshow(amap_n, cmap=ANOM_CMAP, interpolation='nearest')
    pred_s  = 'DEF' if ddpm_pred[si] else 'NOR'
    correct = '✓' if pred_s == label_s else '✗'
    ax_m.set_title(f'{correct} Pred:{pred_s}\nScore={ddpm_test_scores[si]:.3f}',
                   fontsize=9,
                   color=CFG.C_FUSION if pred_s==label_s else CFG.C_DDPM)
    ax_m.axis('off')

fig.text(0.01, 0.37, 'Original',      color='#333333', rotation=90, va='center', fontsize=10, fontweight='bold')
fig.text(0.01, 0.18, 'Recon. Error',  color='#333333', rotation=90, va='center', fontsize=10, fontweight='bold')

fig.suptitle('DDPM — Generative Reconstruction Module Results',
             fontsize=13, fontweight='bold', y=1.01)
fig.savefig(f'{CFG.OUT_DIR}/04_ddpm_results.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 04_ddpm_results.png")

# Generated samples
print("🔄 Generating synthetic normal WBMs via DDIM …")
gen_imgs = ddim_sample(ddpm_unet, (8, 3, CFG.IMG_SIZE, CFG.IMG_SIZE),
                        steps=CFG.DDPM_DDIM_STEPS).cpu().numpy()

apply_pub_style()
fig2, axes2 = plt.subplots(2, 4, figsize=(14, 7), facecolor='white')
fig2.subplots_adjust(hspace=0.05, wspace=0.05)
for i, ax in enumerate(axes2.flatten()):
    disp = (gen_imgs[i, 2] + 1) / 2
    ax.imshow(disp, cmap=WBM_CMAP, interpolation='nearest', vmin=0, vmax=1)
    ax.set_title(f'Gen #{i+1}', fontsize=9, fontweight='bold', pad=4)
    ax.axis('off')
fig2.suptitle('DDPM Generated Normal WBMs (DDIM, 50 steps)',
              fontsize=12, fontweight='bold', y=1.01)
fig2.savefig(f'{CFG.OUT_DIR}/05_ddpm_generated_samples.png',
             bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 05_ddpm_generated_samples.png")


✅ Figure saved → 04_ddpm_results.png
🔄 Generating synthetic normal WBMs via DDIM …
✅ Figure saved → 05_ddpm_generated_samples.png


---
# Part III — Fusion: Structural + Generative Scores

```
score_fused(x) = α · score_EAD(x) + (1−α) · score_DDPM(x)
```

Both modules process each image in a **single pass** — α = 0.5 (equal weight; tunable).


In [22]:
# ── Cell 19: Score Fusion & Optimal Threshold ──────────────────────────────
ALPHA = CFG.FUSION_ALPHA    # 0.5

fused_val = ALPHA * ead_val_scores + (1 - ALPHA) * ddpm_val_scores
fpr_fv, tpr_fv, thr_fv = roc_curve(y_val, fused_val)
opt_thr_fused = thr_fv[np.argmax(tpr_fv - fpr_fv)]

fused_test  = ALPHA * ead_test_scores + (1 - ALPHA) * ddpm_test_scores
fused_pred  = (fused_test >= opt_thr_fused).astype(int)

fused_auroc = roc_auc_score(y_test, fused_test)
fused_ap    = average_precision_score(y_test, fused_test)
fused_f1    = f1_score(y_test, fused_pred)
fused_mcc   = matthews_corrcoef(y_test, fused_pred)
fused_bal   = balanced_accuracy_score(y_test, fused_pred)

print("📊 Fused Module Test Results:")
print(f"   AUROC         : {fused_auroc:.4f}")
print(f"   Avg Precision : {fused_ap:.4f}")
print(f"   F1            : {fused_f1:.4f}")
print(f"   MCC           : {fused_mcc:.4f}")
print(f"   Balanced Acc  : {fused_bal:.4f}")

print(f"\n📊 Comparison (Test Set AUROC):")
print(f"   EfficientAD   : {ead_auroc:.4f}")
print(f"   DDPM          : {ddpm_auroc:.4f}")
print(f"   Fused (α=0.5) : {fused_auroc:.4f}")

# Spatial map fusion
def norm_map(m):
    mn = m.min(axis=(1,2), keepdims=True)
    mx = m.max(axis=(1,2), keepdims=True)
    return (m - mn) / (mx - mn + 1e-8)

ead_maps_n  = norm_map(ead_test_maps)
ddpm_maps_n = norm_map(ddpm_test_maps)
fused_maps  = ALPHA * ead_maps_n + (1 - ALPHA) * ddpm_maps_n
print("\n✅ Spatial anomaly map fusion complete. Shape:", fused_maps.shape)


📊 Fused Module Test Results:
   AUROC         : 0.9118
   Avg Precision : 0.9992
   F1            : 0.9089
   MCC           : 0.1490
   Balanced Acc  : 0.8139

📊 Comparison (Test Set AUROC):
   EfficientAD   : 0.9834
   DDPM          : 0.6411
   Fused (α=0.5) : 0.9118

✅ Spatial anomaly map fusion complete. Shape: (18658, 64, 64)


In [23]:
# ── Cell 20: Comprehensive Metrics + Bootstrap 95% CI ─────────────────────
RNG = np.random.default_rng(CFG.SEED)

def bootstrap_ci(y_true, scores, preds, metric_fn, n_boot=CFG.N_BOOTSTRAP, ci=CFG.CI_LEVEL):
    n, vals = len(y_true), []
    for _ in range(n_boot):
        idx = RNG.integers(0, n, size=n)
        try:    v = metric_fn(y_true[idx], scores[idx])
        except: v = metric_fn(y_true[idx], preds[idx])
        vals.append(v)
    vals = np.array(vals)
    lo = np.percentile(vals, 100*(1-ci)/2)
    hi = np.percentile(vals, 100*(1-(1-ci)/2))
    return float(np.mean(vals)), lo, hi

def all_metrics(y_true, scores, preds, tag=''):
    result = {
        'AUROC'   : roc_auc_score(y_true, scores),
        'AP'      : average_precision_score(y_true, scores),
        'F1'      : f1_score(y_true, preds),
        'MCC'     : matthews_corrcoef(y_true, preds),
        'Bal_Acc' : balanced_accuracy_score(y_true, preds),
    }
    for name, fn, use_s in [
        ('AUROC',   lambda yt,s: roc_auc_score(yt,s),           True),
        ('AP',      lambda yt,s: average_precision_score(yt,s),  True),
        ('F1',      lambda yt,s: f1_score(yt,s),                 False),
        ('MCC',     lambda yt,s: matthews_corrcoef(yt,s),        False),
        ('Bal_Acc', lambda yt,s: balanced_accuracy_score(yt,s),  False),
    ]:
        mu, lo, hi = bootstrap_ci(
            y_true, scores if use_s else preds, preds, fn)
        result[f'{name}_lo'] = lo
        result[f'{name}_hi'] = hi
    if tag:
        print(f"\n{'─'*55}\n  {tag}\n{'─'*55}")
        for m in ['AUROC','AP','F1','MCC','Bal_Acc']:
            print(f"  {m:<10}: {result[m]:.4f}  "
                  f"[{result[f'{m}_lo']:.4f}, {result[f'{m}_hi']:.4f}]  95% CI")
    return result

print("🔄 Computing bootstrap CIs (N=1000) … (~30 s)")
res_ead   = all_metrics(y_test, ead_test_scores,  ead_pred,   tag='EfficientAD (Structural)')
res_ddpm  = all_metrics(y_test, ddpm_test_scores, ddpm_pred,  tag='DDPM (Generative)')
res_fused = all_metrics(y_test, fused_test,       fused_pred, tag='Hybrid Fusion (α=0.5)')
print("\n✅ Bootstrap CI computation complete.")


🔄 Computing bootstrap CIs (N=1000) … (~30 s)

───────────────────────────────────────────────────────
  EfficientAD (Structural)
───────────────────────────────────────────────────────
  AUROC     : 0.9834  [0.9742, 0.9915]  95% CI
  AP        : 0.9998  [0.9998, 0.9999]  95% CI
  F1        : 0.9692  [0.9674, 0.9710]  95% CI
  MCC       : 0.3116  [0.2826, 0.3373]  95% CI
  Bal_Acc   : 0.9337  [0.9130, 0.9532]  95% CI

───────────────────────────────────────────────────────
  DDPM (Generative)
───────────────────────────────────────────────────────
  AUROC     : 0.6411  [0.5945, 0.6857]  95% CI
  AP        : 0.9948  [0.9935, 0.9960]  95% CI
  F1        : 0.7096  [0.7039, 0.7153]  95% CI
  MCC       : 0.0332  [0.0182, 0.0473]  95% CI
  Bal_Acc   : 0.5924  [0.5531, 0.6276]  95% CI

───────────────────────────────────────────────────────
  Hybrid Fusion (α=0.5)
───────────────────────────────────────────────────────
  AUROC     : 0.9118  [0.8920, 0.9293]  95% CI
  AP        : 0.9992  [0.998

In [24]:
# ── Cell 21: McNemar's Statistical Significance Test ──────────────────────

def mcnemar_test(y_true, pred_a, pred_b, label_a='A', label_b='B'):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(( correct_a & ~correct_b).sum())
    c = int((~correct_a &  correct_b).sum())
    if (b + c) == 0:
        chi2_stat, p_val = 0.0, 1.0
    else:
        chi2_stat = (abs(b - c) - 1.0)**2 / (b + c)
        p_val     = 1 - chi2.cdf(chi2_stat, df=1)
    sig = '***' if p_val<0.001 else ('**' if p_val<0.01 else ('*' if p_val<0.05 else 'ns'))
    print(f"  McNemar ({label_a} vs {label_b}): "
          f"b={b}, c={c} | χ²={chi2_stat:.3f} | p={p_val:.4f} {sig}")
    return chi2_stat, p_val

print("📊 McNemar's Statistical Significance Tests:")
mcnemar_test(y_test, ead_pred,  ddpm_pred,  'EfficientAD', 'DDPM')
mcnemar_test(y_test, ead_pred,  fused_pred, 'EfficientAD', 'Fusion')
mcnemar_test(y_test, ddpm_pred, fused_pred, 'DDPM',        'Fusion')
print("\n  Significance: *** p<0.001  ** p<0.01  * p<0.05  ns: not significant")


📊 McNemar's Statistical Significance Tests:
  McNemar (EfficientAD vs DDPM): b=7710, c=461 | χ²=6429.263 | p=0.0000 ***
  McNemar (EfficientAD vs Fusion): b=2329, c=340 | χ²=1480.758 | p=0.0000 ***
  McNemar (DDPM vs Fusion): b=128, c=5388 | χ²=5013.974 | p=0.0000 ***

  Significance: *** p<0.001  ** p<0.01  * p<0.05  ns: not significant


In [25]:
# ── Cell 22: Spatial Anomaly Map Visualisation ─────────────────────────────
apply_pub_style()

tp_idx = np.where((fused_pred==1) & (y_test==1))[0]
tn_idx = np.where((fused_pred==0) & (y_test==0))[0]
fp_idx = np.where((fused_pred==1) & (y_test==0))[0]
fn_idx = np.where((fused_pred==0) & (y_test==1))[0]

show_list = []
for grp, label in [(tp_idx,'TP'),(tn_idx,'TN'),(fp_idx,'FP'),(fn_idx,'FN')]:
    if len(grp): show_list.append((grp[0], label))
while len(show_list) < 5:
    if len(tp_idx): show_list.append((tp_idx[len(show_list) % len(tp_idx)], 'TP'))
    else: break

n_cols = len(show_list)
fig = plt.figure(figsize=(4.5 * n_cols, 17), facecolor='white')
gs  = gridspec.GridSpec(4, n_cols, figure=fig, hspace=0.10, wspace=0.08)

row_titles = ['Original WBM', 'EfficientAD Map', 'DDPM Map', 'Fused Map']
cmaps_     = [WBM_CMAP, ANOM_CMAP, 'magma', 'inferno']

for col, (si, outcome) in enumerate(show_list):
    raw      = T_test[si].numpy()
    orig     = (raw[2] + 1) / 2
    emap     = norm_map(ead_test_maps[si:si+1])[0]
    dmap     = norm_map(ddpm_test_maps[si:si+1])[0]
    fmap_n   = fused_maps[si]
    fmap_n   = (fmap_n - fmap_n.min()) / (fmap_n.max() - fmap_n.min() + 1e-8)
    gt_s     = 'DEF' if y_test[si] else 'NOR'
    col_c    = {
        'TP': CFG.C_FUSION, 'TN': CFG.C_EAD,
        'FP': CFG.C_BASE,   'FN': CFG.C_DDPM
    }[outcome]

    for row, (mdata, cmap_name) in enumerate(
            zip([orig, emap, dmap, fmap_n], cmaps_)):
        ax = fig.add_subplot(gs[row, col])
        im = ax.imshow(mdata, cmap=cmap_name, interpolation='nearest', vmin=0, vmax=1)
        ax.axis('off')
        if row == 0:
            ax.set_title(f'[{outcome}]\nGT: {gt_s}\nScore={fused_test[si]:.3f}',
                         fontsize=9, fontweight='bold', color=col_c, pad=5)
        for spine in ax.spines.values():
            spine.set_visible(True); spine.set_edgecolor(col_c); spine.set_linewidth(1.5)
        if col == 0:
            ax.set_ylabel(row_titles[row], fontsize=9, fontweight='bold', labelpad=4)
            ax.yaxis.set_tick_params(labelleft=False)

# Legend
legend_items = [
    ('TP – Correct Defect', CFG.C_FUSION),
    ('TN – Correct Normal', CFG.C_EAD),
    ('FP – False Alarm',    CFG.C_BASE),
    ('FN – Missed Defect',  CFG.C_DDPM),
]
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, label=l) for l, c in legend_items]
fig.legend(handles=handles, loc='upper right', framealpha=0.9, fontsize=9,
           bbox_to_anchor=(0.99, 0.99))

fig.suptitle('Hybrid Framework — Spatial Anomaly Map Decomposition',
             fontsize=13, fontweight='bold', y=1.005)
fig.savefig(f'{CFG.OUT_DIR}/06_spatial_anomaly_maps.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 06_spatial_anomaly_maps.png")


✅ Figure saved → 06_spatial_anomaly_maps.png


In [26]:
# ── Cell 23: ROC, PR & Calibration Curves ──────────────────────────────────
apply_pub_style()

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor='white')
fig.subplots_adjust(wspace=0.30)

# Three-way ROC
ax = axes[0]
for scores, label, color, ls in [
    (ead_test_scores,  f'EfficientAD  AUROC={ead_auroc:.4f}',  CFG.C_EAD,    '-'),
    (ddpm_test_scores, f'DDPM         AUROC={ddpm_auroc:.4f}',  CFG.C_DDPM,   '--'),
    (fused_test,       f'Fusion       AUROC={fused_auroc:.4f}', CFG.C_FUSION, '-'),
]:
    fpr_, tpr_, _ = roc_curve(y_test, scores)
    ax.plot(fpr_, tpr_, color=color, linewidth=2.2, linestyle=ls, label=label)
    ax.fill_between(fpr_, tpr_, alpha=0.05, color=color)
ax.plot([0,1],[0,1], color='#aaaaaa', linewidth=1.0, linestyle=':', label='Random')
ax.set_title('ROC Curve Comparison', fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1.02)

# PR Comparison
ax = axes[1]
for scores, label, color, ls in [
    (ead_test_scores,  f'EfficientAD  AP={ead_ap:.4f}',  CFG.C_EAD,    '-'),
    (ddpm_test_scores, f'DDPM         AP={ddpm_ap:.4f}',  CFG.C_DDPM,   '--'),
    (fused_test,       f'Fusion       AP={fused_ap:.4f}', CFG.C_FUSION, '-'),
]:
    p_, r_, _ = precision_recall_curve(y_test, scores)
    ax.plot(r_, p_, color=color, linewidth=2.2, linestyle=ls, label=label)
ax.set_title('Precision-Recall Curve Comparison', fontweight='bold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.legend(fontsize=8)

# Reliability / Calibration
ax = axes[2]
n_bins = 10
for scores, label, color in [
    (ead_test_scores,  'EfficientAD', CFG.C_EAD),
    (ddpm_test_scores, 'DDPM',        CFG.C_DDPM),
    (fused_test,       'Fusion',      CFG.C_FUSION),
]:
    edges = np.linspace(0, 1, n_bins + 1)
    bm, bf = [], []
    for lo_b, hi_b in zip(edges[:-1], edges[1:]):
        mask = (scores >= lo_b) & (scores < hi_b)
        if mask.sum() > 0:
            bm.append(scores[mask].mean())
            bf.append(y_test[mask].mean())
    if bm:
        ax.plot(bm, bf, marker='o', markersize=5,
                color=color, linewidth=1.8, label=label)
ax.plot([0,1],[0,1], 'k--', linewidth=1.0, label='Perfect calibration')
ax.set_title('Reliability Diagram (Calibration)', fontweight='bold')
ax.set_xlabel('Mean Predicted Score'); ax.set_ylabel('Fraction of Defectives')
ax.legend(fontsize=8)

fig.suptitle('Hybrid Framework — Comprehensive Curve Analysis',
             fontsize=13, fontweight='bold', y=1.02)
fig.savefig(f'{CFG.OUT_DIR}/07_roc_pr_calibration.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 07_roc_pr_calibration.png")


✅ Figure saved → 07_roc_pr_calibration.png


In [27]:
# ── Cell 24: Publication-Quality Results Table ─────────────────────────────
rows = []
for tag, res, pm in [
    ('EfficientAD (Structural)',  res_ead,   None),
    ('DDPM (Generative)',         res_ddpm,  n_params/1e6),
    ('Hybrid Fusion (α=0.5)',     res_fused, None),
]:
    rows.append({
        'Model'    : tag,
        'AUROC'    : f"{res['AUROC']:.4f} [{res['AUROC_lo']:.4f}–{res['AUROC_hi']:.4f}]",
        'AP'       : f"{res['AP']:.4f} [{res['AP_lo']:.4f}–{res['AP_hi']:.4f}]",
        'F1'       : f"{res['F1']:.4f} [{res['F1_lo']:.4f}–{res['F1_hi']:.4f}]",
        'MCC'      : f"{res['MCC']:.4f} [{res['MCC_lo']:.4f}–{res['MCC_hi']:.4f}]",
        'Bal.Acc'  : f"{res['Bal_Acc']:.4f} [{res['Bal_Acc_lo']:.4f}–{res['Bal_Acc_hi']:.4f}]",
        'Params(M)': f"{pm:.2f}" if pm else '—',
    })

df_results = pd.DataFrame(rows)
print("\n📋 Publication-Quality Results Table (Test Set, 95% CI):")
print(df_results.to_string(index=False))
df_results.to_csv(f'{CFG.OUT_DIR}/08_results_table.csv', index=False)
print(f"\n✅ CSV saved → 08_results_table.csv")

# Figure version of the table
apply_pub_style()
fig, ax = plt.subplots(figsize=(16, 2.8), facecolor='white')
ax.axis('off')

table = ax.table(
    cellText=df_results.values,
    colLabels=df_results.columns,
    cellLoc='center', loc='center', bbox=[0, 0, 1, 1],
)
table.auto_set_font_size(False)
table.set_fontsize(9)

header_color = '#2171b5'
row_colors   = ['#f7fbff', '#ffffff']
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor(header_color)
        cell.set_text_props(color='white', fontweight='bold')
    else:
        cell.set_facecolor(row_colors[r % 2])
        cell.set_text_props(color='#222222')
    cell.set_edgecolor('#cccccc')
    cell.set_linewidth(0.5)

fig.suptitle('Hybrid EfficientAD + DDPM — Results Summary (95% Bootstrap CI)',
             fontsize=11, fontweight='bold', y=1.05)
fig.savefig(f'{CFG.OUT_DIR}/09_results_table_figure.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 09_results_table_figure.png")



📋 Publication-Quality Results Table (Test Set, 95% CI):
                   Model                  AUROC                     AP                     F1                    MCC                Bal.Acc Params(M)
EfficientAD (Structural) 0.9834 [0.9742–0.9915] 0.9998 [0.9998–0.9999] 0.9692 [0.9674–0.9710] 0.3116 [0.2826–0.3373] 0.9337 [0.9130–0.9532]         —
       DDPM (Generative) 0.6411 [0.5945–0.6857] 0.9948 [0.9935–0.9960] 0.7096 [0.7039–0.7153] 0.0332 [0.0182–0.0473] 0.5924 [0.5531–0.6276]      7.83
   Hybrid Fusion (α=0.5) 0.9118 [0.8920–0.9293] 0.9992 [0.9989–0.9994] 0.9089 [0.9058–0.9118] 0.1490 [0.1293–0.1659] 0.8139 [0.7808–0.8458]         —

✅ CSV saved → 08_results_table.csv
✅ Figure saved → 09_results_table_figure.png


---
# Part IV — Baseline Comparison

| Baseline | Key Idea |
|----------|----------|
| **PCA + Mahalanobis** | Top-k PCA; Mahalanobis distance from normal subspace |
| **Simple Autoencoder** | Reconstruction error from a shallow conv AE |
| **Statistical Z-score** | Pixel-wise z-score relative to normal wafer statistics |


In [28]:
# ── Cell 25: Baseline Method Implementations ───────────────────────────────
from sklearn.decomposition import PCA

N_COMPONENTS = 64

# ── Baseline 1: PCA + Mahalanobis ──────────────────────────────────────────
print("🔄 Baseline 1: PCA + Mahalanobis …")
X_train_flat = X_train[:, 2].reshape(len(X_train), -1)
X_val_flat   = X_val[:, 2].reshape(len(X_val),   -1)
X_test_flat  = X_test[:, 2].reshape(len(X_test),  -1)

scaler_pca = StandardScaler()
X_tr_sc = scaler_pca.fit_transform(X_train_flat)
X_va_sc = scaler_pca.transform(X_val_flat)
X_te_sc = scaler_pca.transform(X_test_flat)

pca = PCA(n_components=N_COMPONENTS, random_state=SEED)
pca.fit(X_tr_sc)
Z_tr = pca.transform(X_tr_sc)
Z_te = pca.transform(X_te_sc)

mu_pca  = Z_tr.mean(axis=0)
cov_pca = np.cov(Z_tr.T) + 1e-8 * np.eye(N_COMPONENTS)
cov_inv = np.linalg.pinv(cov_pca)

def mahal_dist(Z, mu, cov_inv):
    diff = Z - mu
    return np.sqrt(np.einsum('ij,jk,ik->i', diff, cov_inv, diff))

pca_raw  = mahal_dist(Z_te, mu_pca, cov_inv)
pca_lo, pca_hi = np.percentile(pca_raw, 1), np.percentile(pca_raw, 99)
pca_test_scores = np.clip((pca_raw - pca_lo) / (pca_hi - pca_lo + 1e-8), 0, 1)
pca_auroc = roc_auc_score(y_test, pca_test_scores)
pca_ap    = average_precision_score(y_test, pca_test_scores)
print(f"   PCA ({N_COMPONENTS} comp) → AUROC: {pca_auroc:.4f}  AP: {pca_ap:.4f}")

# ── Baseline 2: Simple Autoencoder ─────────────────────────────────────────
print("\n🔄 Baseline 2: Simple Autoencoder …")

class SimpleAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 16, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(16,32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32,64, 3, 2, 1), nn.ReLU(),
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64,32, 2, 2), nn.ReLU(),
            nn.ConvTranspose2d(32,16, 2, 2), nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 2, 2), nn.Tanh(),
        )
    def forward(self, x): return self.dec(self.enc(x))

sae     = SimpleAE().to(DEVICE)
sae_opt = torch.optim.Adam(sae.parameters(), lr=1e-3)
sae_ldr = DataLoader(TensorDataset(T_train), batch_size=64, shuffle=True)

for ep in range(20):
    sae.train()
    for (x,) in sae_ldr:
        x = x.to(DEVICE)
        loss = F.mse_loss(sae(x), x)
        sae_opt.zero_grad(); loss.backward(); sae_opt.step()

sae.eval()
sae_raw = []
with torch.no_grad():
    for i in range(0, len(T_test), 64):
        x = T_test[i:i+64].to(DEVICE)
        err = F.mse_loss(sae(x), x, reduction='none').mean(dim=[1,2,3])
        sae_raw.extend(err.cpu().numpy())
sae_raw = np.array(sae_raw)
sae_lo, sae_hi = np.percentile(sae_raw, 1), np.percentile(sae_raw, 99)
sae_test_scores = np.clip((sae_raw - sae_lo) / (sae_hi - sae_lo + 1e-8), 0, 1)
sae_auroc = roc_auc_score(y_test, sae_test_scores)
sae_ap    = average_precision_score(y_test, sae_test_scores)
print(f"   Simple AE (20ep) → AUROC: {sae_auroc:.4f}  AP: {sae_ap:.4f}")

# ── Baseline 3: Z-score ────────────────────────────────────────────────────
print("\n🔄 Baseline 3: Statistical Z-score …")
X_tr_broken  = X_train[:, 2]
pixel_mean   = X_tr_broken.mean(axis=0)
pixel_std    = X_tr_broken.std(axis=0) + 1e-8
z_maps       = np.abs((X_test[:, 2] - pixel_mean) / pixel_std)
z_raw        = z_maps.reshape(len(X_test), -1).mean(axis=1)
z_lo, z_hi   = np.percentile(z_raw, 1), np.percentile(z_raw, 99)
zscore_test_scores = np.clip((z_raw - z_lo) / (z_hi - z_lo + 1e-8), 0, 1)
zscore_auroc = roc_auc_score(y_test, zscore_test_scores)
zscore_ap    = average_precision_score(y_test, zscore_test_scores)
print(f"   Z-score          → AUROC: {zscore_auroc:.4f}  AP: {zscore_ap:.4f}")


🔄 Baseline 1: PCA + Mahalanobis …
   PCA (64 comp) → AUROC: 0.9990  AP: 1.0000

🔄 Baseline 2: Simple Autoencoder …
   Simple AE (20ep) → AUROC: 0.9990  AP: 1.0000

🔄 Baseline 3: Statistical Z-score …
   Z-score          → AUROC: 0.9990  AP: 1.0000


In [29]:
# ── Cell 26: Multi-Model Comparison Bar Chart ──────────────────────────────
apply_pub_style()

model_labels = [
    'Z-score\n(Stat.)',
    'PCA+\nMahal.',
    'Simple\nAutoenc.',
    'EfficientAD\n(Ours)',
    'DDPM\n(Ours)',
    'Hybrid\nFusion (Ours)',
]
auroc_vals = [zscore_auroc, pca_auroc, sae_auroc,
              ead_auroc,    ddpm_auroc, fused_auroc]
ap_vals    = [
    average_precision_score(y_test, zscore_test_scores),
    pca_ap, sae_ap, ead_ap, ddpm_ap, fused_ap,
]
bar_colors = [CFG.C_BASE]*3 + [CFG.C_EAD, CFG.C_DDPM, CFG.C_FUSION]
n_models   = len(model_labels)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='white')
fig.subplots_adjust(wspace=0.28)

for ax, vals, ylabel, title in [
    (axes[0], auroc_vals, 'AUROC',            'Image-Level AUROC Comparison'),
    (axes[1], ap_vals,    'Average Precision', 'Average Precision Comparison'),
]:
    bars = ax.bar(range(n_models), vals, color=bar_colors,
                  edgecolor='white', linewidth=0.5, width=0.55)
    ax.set_xticks(range(n_models))
    ax.set_xticklabels(model_labels, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.set_ylim(max(0, min(vals)*0.94), 1.02)
    ax.axhline(1.0, color='#333333', linewidth=0.7, linestyle='--', alpha=0.5)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.003,
                f'{val:.4f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold')

    # Divider between baselines and proposed
    ax.axvline(2.5, color='#555555', linewidth=1.2, linestyle='--', alpha=0.7)
    ax.text(1.0,  max(vals)*0.95, 'Baselines', ha='center',
            fontsize=9, color='#555555', style='italic')
    ax.text(4.0,  max(vals)*0.95, 'Proposed',  ha='center',
            fontsize=9, color=CFG.C_FUSION, fontweight='bold', style='italic')

fig.suptitle('WM-38K Anomaly Detection — Model Comparison (Test Set)',
             fontsize=13, fontweight='bold', y=1.02)
fig.savefig(f'{CFG.OUT_DIR}/10_comparison_bars.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 10_comparison_bars.png")


✅ Figure saved → 10_comparison_bars.png


In [30]:
# ── Cell 27: t-SNE Feature Space Visualisation ─────────────────────────────
from sklearn.manifold import TSNE

print("🔄 Extracting features for t-SNE (500 samples) …")
N_TSNE   = min(500, len(y_test))
tsne_idx = np.random.choice(len(y_test), N_TSNE, replace=False)

student_net.eval()
ead_feats = []
with torch.no_grad():
    for si in range(0, N_TSNE, 64):
        batch = T_test[tsne_idx[si:si+64]].to(DEVICE)
        feat  = student_net(batch)
        feat  = F.adaptive_avg_pool2d(feat, 1).squeeze(-1).squeeze(-1)
        ead_feats.append(feat.cpu().numpy())

ead_feats = np.concatenate(ead_feats, axis=0)
y_tsne    = y_test[tsne_idx]

print(f"   Feature shape: {ead_feats.shape}")
print("🔄 Running t-SNE …")
tsne_embed = TSNE(n_components=2, perplexity=40, random_state=SEED,
                  n_iter=1000, learning_rate='auto').fit_transform(ead_feats)

apply_pub_style()
fig, ax = plt.subplots(figsize=(8, 7), facecolor='white')

for lv, color, name in [(0, CFG.C_NORMAL, 'Normal'), (1, CFG.C_DEFECT, 'Defective')]:
    mask = (y_tsne == lv)
    ax.scatter(tsne_embed[mask, 0], tsne_embed[mask, 1],
               c=color, s=18, alpha=0.75, edgecolors='none',
               label=f'{name} (n={mask.sum()})')

ax.set_title('t-SNE of EfficientAD Student Features (N=500)',
             fontweight='bold', pad=10)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(fontsize=9)

fig.savefig(f'{CFG.OUT_DIR}/11_tsne_features.png',
            bbox_inches='tight', dpi=CFG.FIG_DPI, facecolor='white')
plt.show()
print("✅ Figure saved → 11_tsne_features.png")


🔄 Extracting features for t-SNE (500 samples) …
   Feature shape: (500, 384)
🔄 Running t-SNE …
✅ Figure saved → 11_tsne_features.png


In [31]:
# ── Cell 28: Final Summary ─────────────────────────────────────────────────
print("=" * 65)
print("  HYBRID EFFICIENTAD + DDPM FRAMEWORK — FINAL SUMMARY")
print("=" * 65)
print(f"\n  Dataset    : WM-38K  ({N:,} wafers, 38 classes, {CFG.IMG_SIZE}×{CFG.IMG_SIZE} px)")
print(f"  Task       : One-class anomaly detection (normal vs defective)")
print(f"  Train set  : {len(X_train):,} normal wafers")
print(f"  Test  set  : {len(X_test):,} wafers "
      f"({y_test.sum():,} def / {(y_test==0).sum():,} nor)\n")

print("  ── MODULE 1: EfficientAD (Structural) ─────────────────")
print(f"     Architecture  : Multi-Scale PDN + ResNet-18 Teacher")
print(f"     Training loss : {EAD_HISTORY['loss'][-1]:.5f}  (target {CFG.EAD_TARGET_LOSS})")
print(f"     AUROC (test)  : {ead_auroc:.4f}  (target 0.9987)")
print(f"     F1            : {ead_f1:.4f}  |  MCC: {ead_mcc:.4f}")

print("\n  ── MODULE 2: DDPM (Generative) ─────────────────────────")
print(f"     Architecture  : U-Net ε-predictor, cosine schedule, EMA")
print(f"     Parameters    : {n_params/1e6:.2f} M  (target {CFG.DDPM_TARGET_PARAMS} M)")
print(f"     Epochs        : {CFG.DDPM_EPOCHS}")
print(f"     Training loss : {DDPM_HISTORY['loss'][-1]:.5f}  (target {CFG.DDPM_TARGET_LOSS})")
print(f"     Sampling rate : {sampling_rate:.2f} it/s  (target 45.34 it/s)")
print(f"     AUROC (test)  : {ddpm_auroc:.4f}")
print(f"     F1            : {ddpm_f1:.4f}  |  MCC: {ddpm_mcc:.4f}")

print("\n  ── FUSION: Hybrid Score (α=0.5) ─────────────────────────")
print(f"     AUROC (test)  : {fused_auroc:.4f}")
print(f"     AP (test)     : {fused_ap:.4f}")
print(f"     F1            : {fused_f1:.4f}")
print(f"     MCC           : {fused_mcc:.4f}")
print(f"     Balanced Acc  : {fused_bal:.4f}")

print("\n  ── BASELINES ────────────────────────────────────────────")
print(f"     Z-score       : AUROC {zscore_auroc:.4f}")
print(f"     PCA (k={N_COMPONENTS})    : AUROC {pca_auroc:.4f}")
print(f"     Simple AE     : AUROC {sae_auroc:.4f}")

print("\n  ── SAVED FIGURES ────────────────────────────────────────")
for f in [
    "01_eda_comprehensive.png        — EDA overview",
    "02_ead_training_curves.png      — EfficientAD loss curves",
    "03_ead_results.png              — ROC / score dist / anomaly maps",
    "04_ddpm_results.png             — DDPM ROC / recon visualisation",
    "05_ddpm_generated_samples.png   — DDIM-sampled normal WBMs",
    "06_spatial_anomaly_maps.png     — TP/TN/FP/FN spatial decomp",
    "07_roc_pr_calibration.png       — Three-way curves",
    "08_results_table.csv            — Full metrics with 95% CI",
    "09_results_table_figure.png     — Publication table figure",
    "10_comparison_bars.png          — Baseline comparison chart",
    "11_tsne_features.png            — t-SNE feature visualisation",
]:
    print(f"     {f}")

print(f"\n✅ All outputs saved to: {CFG.OUT_DIR}")
print("=" * 65)


  HYBRID EFFICIENTAD + DDPM FRAMEWORK — FINAL SUMMARY

  Dataset    : WM-38K  (38,015 wafers, 38 classes, 64×64 px)
  Task       : One-class anomaly detection (normal vs defective)
  Train set  : 700 normal wafers
  Test  set  : 18,658 wafers (18,508 def / 150 nor)

  ── MODULE 1: EfficientAD (Structural) ─────────────────
     Architecture  : Multi-Scale PDN + ResNet-18 Teacher
     Training loss : 0.16382  (target 0.01564)
     AUROC (test)  : 0.9834  (target 0.9987)
     F1            : 0.9692  |  MCC: 0.3116

  ── MODULE 2: DDPM (Generative) ─────────────────────────
     Architecture  : U-Net ε-predictor, cosine schedule, EMA
     Parameters    : 7.83 M  (target 7.16 M)
     Epochs        : 30
     Training loss : 0.04879  (target 0.01633)
     Sampling rate : 13.22 it/s  (target 45.34 it/s)
     AUROC (test)  : 0.6411
     F1            : 0.7096  |  MCC: 0.0332

  ── FUSION: Hybrid Score (α=0.5) ─────────────────────────
     AUROC (test)  : 0.9118
     AP (test)     : 0.9992
   